> **English archival edition.** Saved outputs are historical observations, preserved rather than rerun. Chinese text has been translated. Use a new run label or output directory: translated source hashes differ from historical checkpoints.

# v6.1 ·  Locality of selection  top-10  intervention 

 in  Colab  Select with  baseline  Same.  A100（ Last Round As  A100-SXM4-80GB）， Click 「 Run All 」。 If you install a grid tip to restart, then click after the session is restarted 「 Run All 」。 Local experiments have been launched in this edition. 

 Reuse  Drive  Completed  288   baseline  Unit. In.  24  Before comparing requests to original papers  10  Random head matching three groups by layer  10  heads. Add to each group  144  Artisan autonomy  Agent  Tracks and pairings, in total  576  A new unit. First look.  top-10  Could it be more effective than random comparisons?  Clex  to  C1  item-selection advantage reduction, while retaining strict task success under the neutral condition. 

 Keep the old one.  main  Catalogues. You can run at breakpoints in the same environment.  local COMPLETE [864, 864]。 The last one will verify the results.  ZIP、 Sync  Drive， Released.  GPU。

In [1]:
import sys, subprocess, importlib.metadata as v5_metadata
if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError(' Fixed-dependent combination needs  Python 3.10–3.13。')
V6_REQUIRED_VERSIONS = {
    'transformer-lens': '2.17.0', 'transformers': '4.57.1', 'huggingface-hub': '0.36.0',
    'numpy': '2.1.3' if sys.version_info[:2] == (3, 13) else '1.26.4',
    'pandas': '2.2.3', 'plotly': '6.1.2', 'scipy': '1.15.3',
    'scikit-learn': '1.6.1', 'datasets': '3.6.0', 'pyarrow': '20.0.0',
}
V6_INSTALL_PROBLEMS = []
for v5_name, v5_version in V6_REQUIRED_VERSIONS.items():
    try:
        v5_installed = v5_metadata.version(v5_name)
    except v5_metadata.PackageNotFoundError:
        v5_installed = None
    if v5_installed != v5_version:
        V6_INSTALL_PROBLEMS.append((v5_name, v5_installed, v5_version))
if V6_INSTALL_PROBLEMS:
    print(' The original experimental fixed version will be installed: ', V6_INSTALL_PROBLEMS)
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    *[f'{name}=={version}' for name, version in V6_REQUIRED_VERSIONS.items()], 'tqdm'], check=True)
    raise SystemExit(' Installation complete. Please restart  Colab runtime， Runs from this block; versions are not repeated after matching. ')
print(' Fixed reliance matches, skips installation. ')


 Fixed reliance matches, skips installation. 


In [2]:
#  Run this block after restarting: check if the installation is currently in place  Python  Environment. 
import sys
import importlib.metadata as metadata
import importlib.util

print("Python:", sys.version.split()[0])
print(" Interpreter :", sys.executable)
if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError(" Yes.  Python 3.10–3.13； Please re-implement the installment block after running. ")

expected_versions = {
    "transformer-lens": "2.17.0", "transformers": "4.57.1",
    "huggingface-hub": "0.36.0",
    "numpy": "2.1.3" if sys.version_info[:2] == (3, 13) else "1.26.4",
    "pandas": "2.2.3",
    "plotly": "6.1.2", "scipy": "1.15.3", "scikit-learn": "1.6.1",
    "datasets": "3.6.0", "pyarrow": "20.0.0",
}
problems = []
for package, expected in expected_versions.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    print(f"{package}: {installed if installed is not None else ' not installed '} ( Yes.  {expected})")
    if installed != expected:
        problems.append(f"{package}:  current  {installed}， Yes.  {expected}")

for package in ("torch", "tqdm"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        problems.append(f"{package}:  not installed ")

try:
    torch_version = metadata.version("torch")
    torch_major_minor = tuple(int(part) for part in torch_version.split("+")[0].split(".")[:2])
    if torch_major_minor < (2, 6):
        problems.append(f"torch:  current  {torch_version}， Yes.  >=2.6")
except metadata.PackageNotFoundError:
    pass  #  The record is missing. 
except ValueError:
    problems.append(" Cannot parse  torch  Version, please check the current version  Colab  Run when. ")

if importlib.util.find_spec("transformer_lens") is None:
    problems.append(" current  Python  Cannot find  transformer_lens  module ")
if problems:
    raise RuntimeError(
        " Reliance check failed: \n- " + "\n- ".join(problems)
        + "\n Run the first cell and inspect its complete installation log. After successful installation, restart the runtime, "
          " Go on from this block. "
    )
print(" Reliance on a version check pass; continues to run the third code block. ")


Python: 3.13.16
 Interpreter : /usr/bin/python3
transformer-lens: 2.17.0 ( Yes.  2.17.0)
transformers: 4.57.1 ( Yes.  4.57.1)
huggingface-hub: 0.36.0 ( Yes.  0.36.0)
numpy: 2.1.3 ( Yes.  2.1.3)
pandas: 2.2.3 ( Yes.  2.2.3)
plotly: 6.1.2 ( Yes.  6.1.2)
scipy: 1.15.3 ( Yes.  1.15.3)
scikit-learn: 1.6.1 ( Yes.  1.6.1)
datasets: 3.6.0 ( Yes.  3.6.0)
pyarrow: 20.0.0 ( Yes.  20.0.0)
torch: 2.11.0+cu130
tqdm: 4.67.3
 Reliance on a version check pass; continues to run the third code block. 


In [3]:
#  Formal Import: Keep the whole package if there is still an error report here  traceback  In the past few years, many of the worlds leading players have been able to identify specific conflicts. 
import gc, json, math, os, random, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
from scipy import stats
from tqdm.auto import tqdm
from transformer_lens import HookedTransformer, utils
from huggingface_hub import notebook_login, get_token, model_info
import importlib.metadata as metadata
from IPython.display import display

V6_PACKAGE_NAMES = ["torch", "transformer-lens", "transformers", "huggingface-hub", "numpy", "pandas", "datasets", "pyarrow", "scipy", "scikit-learn", "plotly"]
V6_PACKAGE_VERSIONS = {name: metadata.version(name) for name in V6_PACKAGE_NAMES}
print(V6_PACKAGE_VERSIONS)
print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print(" Reliance on import success. ")


{'torch': '2.11.0+cu130', 'transformer-lens': '2.17.0', 'transformers': '4.57.1', 'huggingface-hub': '0.36.0', 'numpy': '2.1.3', 'pandas': '2.2.3', 'datasets': '3.6.0', 'pyarrow': '20.0.0', 'scipy': '1.15.3', 'scikit-learn': '1.6.1', 'plotly': '6.1.2'}
Python: 3.13.16
CUDA available: True
 Reliance on import success. 


## 1 ·  Settings 

 Configured complete local experiments. Maintained.  RUN_LABEL  Consistent with the original experiment to read completed checkpoints. 

In [4]:
from google.colab import drive
from pathlib import Path
import hashlib, re
drive.mount("/content/drive")
RUN_LABEL = "main"            #  The same experiment remains unchanged. 
WORLDS_PER_CALL = None        # None =  all;positive integer limits each  arm  Number of new scenarios 
RUN_LOCAL = True              #  run  top-10  A random head with three groups 
AUTO_DISCONNECT = True        #  Only complete results at the requested stage  ZIP  Verifyed and implemented after synchronisation 
if not re.fullmatch(r"[A-Za-z0-9_-]{1,48}",RUN_LABEL):
    raise ValueError("Invalid RUN_LABEL")
if WORLDS_PER_CALL is not None and (type(WORLDS_PER_CALL) is not int or WORLDS_PER_CALL<=0):
    raise ValueError("WORLDS_PER_CALL must be positive or None")
BASE_DIR = Path("/content/drive/MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v6p1_aligned_sku")/("pilot-"+RUN_LABEL)
REPORT = RESULTS_ZIP = None
REQUESTED_FINISHED = False
V6_CFG = dict(experiment_version="v6_truthful_binding_1",model_name="meta-llama/Llama-3.1-8B",
              model_revision="d04e592bb4f6aa9cfee91e2e20afa771667e1d4b", dtype="bfloat16",
              context_window=4096,max_prompt_tokens=3072,actor_max_new_tokens=128,writer_max_new_tokens=384,
              max_actions=6,use_cache=True,hook_kind="z",data_seed=961103)
print(" Output Directory ", BASE_DIR)


Mounted at /content/drive
 Output Directory  /content/drive/MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v6p1_aligned_sku/pilot-main


## 2 ·  Original executor, saved model responses and experimental realization 

 Archiving part decode only  JSON  Data. The following cells are loaded with  v6  consistent model, parser, tools, and scoring code. This version neither searches heads nor runs  v6  All the way.  80  Head away. 


In [ ]:
# Authenticate before accessing gated Hugging Face model files.
from google.colab import userdata
from huggingface_hub import login
try:
    _archive_hf_token = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError("Set HF_TOKEN in Colab Secrets and enable notebook access.") from None
if not isinstance(_archive_hf_token, str) or not _archive_hf_token.strip():
    raise RuntimeError("HF_TOKEN is empty. Set it in Colab Secrets and enable notebook access.")
login(token=_archive_hf_token.strip(), add_to_git_credential=False)
del _archive_hf_token


In [5]:
import linecache
V6_CORE_SOURCE = r'''import copy, gc, hashlib, json, math, os, random, re, time
from pathlib import Path


def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)

def stable_hash(obj):
    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def normalize_heads(heads, n_layers, n_heads):
    result = sorted(set((int(layer), int(head)) for layer, head in heads))
    if any(not (0 <= l < n_layers and 0 <= h < n_heads) for l, h in result):
        raise ValueError("Head coordinates out of bounds")
    return result

def ablation_hooks(model, heads=(), drop_mask=None, hook_kind="z"):
    """Mask query-head outputs, including Llama GQA; never mask KV-head indices."""
    if hook_kind not in {"z", "result"}:
        raise ValueError("hook_kind must be z or result")
    if drop_mask is not None and tuple(drop_mask.shape) != (model.cfg.n_layers, model.cfg.n_heads):
        raise ValueError("Drop mask shape must be [layers, query heads]")
    selected = normalize_heads(heads, model.cfg.n_layers, model.cfg.n_heads)
    if drop_mask is not None and selected:
        raise ValueError("Specify hard heads OR differentiable drop mask")
    by_layer = {}
    for layer, head in selected:
        by_layer.setdefault(layer, []).append(head)
    layers = range(model.cfg.n_layers) if drop_mask is not None else sorted(by_layer)
    hooks = []
    for layer in layers:
        def mask_output(value, hook, layer=layer):
            if drop_mask is not None:
                keep = (1 - drop_mask[layer]).to(device=value.device, dtype=value.dtype)
            else:
                keep = torch.ones(model.cfg.n_heads, device=value.device, dtype=value.dtype)
                keep[by_layer[layer]] = 0
            return value * keep.view(1, 1, -1, 1)
        hooks.append((f"blocks.{layer}.attn.hook_{hook_kind}", mask_output))
    return hooks

class TLBackend:
    def __init__(self, model, cfg):
        self.model = model
        self.tokenizer = model.tokenizer
        self.cfg = dict(cfg)
        self.device = next(model.parameters()).device
        self.hook_kind = cfg.get("hook_kind", "z")
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        model.eval()
        model.set_use_attn_result(self.hook_kind == "result")

    def encode(self, text):
        # One unpadded sequence per forward: no padding-mask/last-position ambiguity.
        return self.tokenizer.encode(text, add_special_tokens=True)

    def tokens(self, text):
        ids = self.encode(text)
        limit = min(int(self.cfg.get("max_prompt_tokens", 3072)), int(self.model.cfg.n_ctx))
        if len(ids) > limit:
            raise ValueError(f"Prompt has {len(ids)} tokens, limit={limit}; evidence is never silently truncated")
        return torch.tensor([ids], dtype=torch.long, device=self.device)

    def one_token_id(self, full_prefix, value):
        before, after = self.encode(full_prefix), self.encode(full_prefix + value)
        if len(after) != len(before) + 1 or after[:-1] != before:
            return None
        token = after[-1]
        if token in set(self.tokenizer.all_special_ids):
            return None
        return token

    def margin(self, snapshot, heads=(), drop_mask=None):
        gold = self.one_token_id(snapshot["prompt"], snapshot["gold"])
        wrong = self.one_token_id(snapshot["prompt"], snapshot["distractor"])
        if gold is None or wrong is None or gold == wrong:
            raise ValueError("Snapshot candidates are not two distinct exact-prefix single tokens")
        hooks = ablation_hooks(self.model, heads, drop_mask, self.hook_kind)
        with self.model.hooks(fwd_hooks=hooks):
            logits = self.model(self.tokens(snapshot["prompt"]), return_type="logits")
        # Logit difference equals log P(gold)-log P(distractor).
        return (logits[0, -1, gold] - logits[0, -1, wrong]).float()

    def generate(self, prompt, role="actor", heads=(), max_new_tokens=None, use_cache=True):
        from transformer_lens.past_key_value_caching import HookedTransformerKeyValueCache
        if role not in {"actor", "writer"}:
            raise ValueError("role must be actor or writer")
        budget = int(max_new_tokens or self.cfg[f"{role}_max_new_tokens"])
        prefix = self.tokens(prompt)
        if prefix.shape[1] + budget > int(self.model.cfg.n_ctx):
            raise ValueError("Prompt plus generation budget exceeds model context")
        cache = HookedTransformerKeyValueCache.init_cache(self.model.cfg, self.device, batch_size=1) if use_cache else None
        generated = []
        sequence = prefix
        stop = "max_new_tokens"
        started = time.perf_counter()
        hooks = ablation_hooks(self.model, heads, hook_kind=self.hook_kind)
        # Fresh cache for every call, same intervention for prefill AND decode.
        with torch.inference_mode(), self.model.hooks(fwd_hooks=hooks):
            for step in range(budget):
                tokens = sequence if step == 0 or not use_cache else sequence[:, -1:]
                logits = self.model(tokens, past_kv_cache=cache, return_type="logits")
                next_id = int(logits[0, -1].argmax().item())
                if next_id == self.tokenizer.eos_token_id:
                    stop = "eos"
                    break
                generated.append(next_id)
                sequence = torch.cat([sequence, torch.tensor([[next_id]], device=self.device)], dim=1)
                decoded = self.tokenizer.decode(generated, skip_special_tokens=False)
                parsed, error = parse_json_output(decoded, dict if role == "actor" else list)
                if error is None:
                    stop = "complete_json"
                    break
        text = self.tokenizer.decode(generated, skip_special_tokens=False)
        del cache
        return {"text": text, "token_count": len(generated), "prompt_tokens": int(prefix.shape[1]),
                "seconds": time.perf_counter() - started, "stop_reason": stop}

def _json_no_duplicates(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key}")
        result[key] = value
    return result

def parse_json_output(text, expected_type):
    """Strictly parse one JSON object/list; never repair factual values.

    A single enclosing Markdown fence is tolerated. Trailing explanations,
    duplicate keys, NaN/Infinity, and mismatched types are explicit failures.
    """
    if not isinstance(text, str):
        return None, "output_not_text"
    candidate = text.strip()
    if candidate.startswith("```"):
        match = re.fullmatch(r"```(?:json)?\s*\n?(.*?)\n?```", candidate, flags=re.DOTALL | re.IGNORECASE)
        if match is None:
            return None, "invalid_json_fence"
        candidate = match.group(1).strip()
    try:
        value = json.loads(candidate, object_pairs_hook=_json_no_duplicates,
                           parse_constant=lambda token: (_ for _ in ()).throw(ValueError(f"nonfinite JSON: {token}")))
    except (ValueError, TypeError, json.JSONDecodeError) as exc:
        return None, f"invalid_json:{str(exc)[:160]}"
    if not isinstance(value, expected_type):
        return None, f"wrong_json_type:expected_{getattr(expected_type, '__name__', str(expected_type))}"
    return value, None

class OrderEnvironment:
    """A local, deterministic world; it knows inventory, not customer truth."""

    def __init__(self, initial_state):
        self.state = copy.deepcopy(initial_state)
        self.state.setdefault("stock", {})
        self.state.setdefault("reservations", {})
        self.state.setdefault("orders", [])
        self.state.setdefault("next_reservation", 1)
        self.state.setdefault("next_order", 1)
        if any(type(v) is not int or v < 0 for v in self.state["stock"].values()):
            raise ValueError("stock quantities must be nonnegative integers")
        self.tool_log = []

    def snapshot(self):
        return copy.deepcopy(self.state)

    def step(self, action):
        observation = self._execute(action)
        self.tool_log.append({"action": copy.deepcopy(action), "observation": copy.deepcopy(observation)})
        return copy.deepcopy(observation)

    def _execute(self, action):
        if not isinstance(action, dict):
            return {"ok": False, "error": "action_must_be_object"}
        tool = action.get("tool")
        schemas = {
            "check_stock": {"tool", "item"},
            "reserve_item": {"tool", "item"},
            "create_order": {"tool", "customer", "item", "city", "reservation"},
        }
        if not isinstance(tool, str) or tool not in schemas:
            return {"ok": False, "error": "unknown_tool"}
        if set(action) != schemas[tool] or any(not isinstance(v, str) or not v for v in action.values()):
            return {"ok": False, "error": "invalid_arguments", "tool": tool}
        item = action["item"]
        if item not in self.state["stock"]:
            return {"ok": False, "error": "unknown_item", "item": item}
        if tool == "check_stock":
            return {"ok": True, "item": item, "available": self.state["stock"][item]}
        if tool == "reserve_item":
            if self.state["stock"][item] <= 0:
                return {"ok": False, "error": "out_of_stock", "item": item}
            reservation = f"R{self.state['next_reservation']:04d}"
            self.state["next_reservation"] += 1
            self.state["stock"][item] -= 1
            self.state["reservations"][reservation] = {"item": item, "consumed": False}
            return {"ok": True, "item": item, "reservation": reservation}
        reservation = self.state["reservations"].get(action["reservation"])
        if reservation is None or reservation["consumed"] or reservation["item"] != item:
            return {"ok": False, "error": "invalid_reservation"}
        # No oracle city/customer correction: a semantically wrong order can
        # succeed as a tool call. The held-out evaluator scores it afterwards.
        order = {"order_id": f"O{self.state['next_order']:04d}",
                 "customer": action["customer"], "item": item,
                 "city": action["city"], "reservation": action["reservation"]}
        self.state["next_order"] += 1
        self.state["orders"].append(order)
        reservation["consumed"] = True
        return {"ok": True, "order": copy.deepcopy(order)}

def runtime_smoke(backend):
    """Small actual-model interface checks; no scientific outcome is assumed."""
    model = backend.model
    probe = backend.tokens('A short interface check: the city is')
    head = [(min(1, model.cfg.n_layers - 1), 0)]
    original_kind = backend.hook_kind
    try:
        with torch.no_grad():
            model.set_use_attn_result(False)
            with model.hooks(fwd_hooks=ablation_hooks(model, head, hook_kind="z")):
                z_logits = model(probe, return_type="logits")[0, -1].float()
            model.set_use_attn_result(True)
            with model.hooks(fwd_hooks=ablation_hooks(model, head, hook_kind="result")):
                result_logits = model(probe, return_type="logits")[0, -1].float()
        max_difference = float((z_logits - result_logits).abs().max())
        if not torch.allclose(z_logits, result_logits, atol=0.25, rtol=0.02):
            raise RuntimeError(f"z/result ablation disagreement (max={max_difference}); inspect precision before search")
        model.set_use_attn_result(original_kind == "result")
        with torch.enable_grad():
            mask_logits = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=backend.device, requires_grad=True)
            drop = torch.sigmoid(mask_logits)
            with model.hooks(fwd_hooks=ablation_hooks(model, drop_mask=drop, hook_kind=original_kind)):
                score = model(probe, return_type="logits")[0, -1, :16].float().sum()
            score.backward()
            gradient_norm = float(mask_logits.grad.norm())
            if not math.isfinite(gradient_norm) or gradient_norm == 0:
                raise RuntimeError("Mask gradient is zero or nonfinite")
        # Different cache paths may expose tiny dtype differences. A mismatch
        # is a failed software check, never evidence of an intervention effect.
        generation_prompt = 'Complete one JSON object.\n{"check":'
        cache_agreement = None
        if backend.cfg.get("use_cache", True):
            cached = backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=True)
            uncached = backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=False)
            cache_agreement = cached["text"] == uncached["text"]
            if not cache_agreement:
                raise RuntimeError("Cached/uncached greedy text differs. Inspect precision or start an explicitly uncached run (use_cache=False); do not silently continue.")
        else:
            backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=False)
        return {"passed": True, "z_result_max_abs_difference": max_difference,
                "mask_gradient_norm": gradient_norm, "cache_text_agreement": cache_agreement,
                "cache_test_skipped": not backend.cfg.get("use_cache", True),
                "cuda_peak_gb": torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None}
    finally:
        backend.hook_kind = original_kind
        model.set_use_attn_result(original_kind == "result")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
'''
V6_RUNTIME_SOURCE = r'''"""Independent, pinned v4 runtime with checked live-model reuse.

No function in this module loads weights until v4_load_runtime is called.
The original backend/tools are supplied verbatim in the same isolated namespace.
"""
import ast as v4_ast
import copy as v4_copy
import hashlib as v4_hashlib
import importlib.metadata as v4_metadata
import json as v4_json
from pathlib import Path as V4Path
import sys as v4_sys
import types as v4_types


V4_MODEL_NAME = "meta-llama/Llama-3.1-8B"
V4_MODEL_REVISION = "d04e592bb4f6aa9cfee91e2e20afa771667e1d4b"
V4_PACKAGE_NAMES = (
    "torch", "transformer-lens", "transformers", "huggingface-hub", "numpy",
    "pandas", "datasets", "pyarrow", "scipy", "scikit-learn", "plotly",
)
V4_FIXED_CONFIG = {
    "model_name": V4_MODEL_NAME, "model_revision": V4_MODEL_REVISION,
    "dtype": "bfloat16", "context_window": 4096, "max_prompt_tokens": 3072,
    "actor_max_new_tokens": 128, "writer_max_new_tokens": 384,
    "max_actions": 6, "use_cache": True, "hook_kind": "z",
}


def v4_backend_contract(cfg):
    """Only settings that can change the original generation implementation."""
    keys = ("context_window", "max_prompt_tokens", "actor_max_new_tokens",
            "writer_max_new_tokens", "use_cache", "hook_kind", "dtype", "model_name")
    if not isinstance(cfg, dict) or any(key not in cfg for key in keys):
        raise ValueError("Incomplete backend generation contract")
    return {key: cfg[key] for key in keys}


def v4_code_state(code):
    """Executable-code identity independent of notebook filename/line numbers."""
    def v4_constant(value):
        if isinstance(value, v4_types.CodeType):
            return {"code": v4_code_state(value)}
        if isinstance(value, bytes):
            return {"bytes": value.hex()}
        if isinstance(value, tuple):
            return {"tuple": [v4_constant(item) for item in value]}
        if isinstance(value, frozenset):
            return {"frozenset": sorted((v4_constant(item) for item in value), key=repr)}
        if value is None or isinstance(value, (bool, int, float, str)):
            return value
        return {"type": type(value).__name__, "repr": repr(value)}
    return {
        "bytecode": code.co_code.hex(), "constants": [v4_constant(x) for x in code.co_consts],
        "names": code.co_names, "variables": code.co_varnames,
        "free": code.co_freevars, "cells": code.co_cellvars,
        "args": code.co_argcount, "posonly": code.co_posonlyargcount,
        "kwonly": code.co_kwonlyargcount, "flags": code.co_flags & 0xffff,
        "exception_table": getattr(code, "co_exceptiontable", b"").hex(),
    }


def v4_definition_state(value):
    if isinstance(value, type):
        methods = {}
        for name, member in vars(value).items():
            member = member.__func__ if isinstance(member, (staticmethod, classmethod)) else member
            if isinstance(member, v4_types.FunctionType):
                methods[name] = v4_definition_state(member)
        return {"class": value.__name__, "bases": [base.__name__ for base in value.__bases__],
                "methods": methods}
    if isinstance(value, v4_types.MethodType):
        value = value.__func__
    if isinstance(value, v4_types.FunctionType):
        return {"code": v4_code_state(value.__code__),
                "defaults": v4_copy.deepcopy(value.__defaults__),
                "kwdefaults": v4_copy.deepcopy(value.__kwdefaults__)}
    return v4_copy.deepcopy(value)


def v4_source_exports(source):
    names = []
    for node in v4_ast.parse(source).body:
        if isinstance(node, (v4_ast.FunctionDef, v4_ast.ClassDef)):
            names.append(node.name)
        elif isinstance(node, v4_ast.Assign):
            names.extend(target.id for target in node.targets if isinstance(target, v4_ast.Name))
    return names


def v4_check_sources(core_source, runtime_source):
    """Check live definitions against the bundled sources, without inference."""
    if not isinstance(core_source, str) or not core_source.strip() or not isinstance(runtime_source, str) or not runtime_source.strip():
        raise ValueError("Both exact runtime source strings are required")
    expected = dict(globals())
    names = []
    for label, source in (("core", core_source), ("runtime", runtime_source)):
        exec(compile(source, "<v4-expected-" + label + ">", "exec", dont_inherit=True), expected)
        names.extend(v4_source_exports(source))
    if len(names) != len(set(names)):
        raise RuntimeError("Bundled core/runtime definitions overlap")
    states = {name: v4_definition_state(expected[name]) for name in names}
    for name in names:
        if name not in globals() or stable_hash(v4_definition_state(globals()[name])) != stable_hash(states[name]):
            raise RuntimeError("Live definition does not match bundled source: " + name)
    return states


def v4_check_packages(package_versions):
    if not isinstance(package_versions, dict) or set(package_versions) != set(V4_PACKAGE_NAMES):
        raise ValueError("Use the original complete PACKAGE_VERSIONS schema")
    actual = {name: v4_metadata.version(name) for name in V4_PACKAGE_NAMES}
    if actual != package_versions or str(torch.__version__) != actual["torch"]:
        raise RuntimeError("Installed/imported packages differ from the supplied package identity")
    for name, required in (("transformer-lens", "2.17.0"), ("transformers", "4.57.1"),
                           ("huggingface-hub", "0.36.0")):
        if actual[name] != required:
            raise RuntimeError("Original model stack required: " + name + "==" + required)
    return actual


def v4_plain_state(value):
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    if isinstance(value, dict):
        return {str(key): v4_plain_state(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [v4_plain_state(item) for item in value]
    return {"type": type(value).__module__ + "." + type(value).__qualname__, "repr": str(value)}


def v4_tokenizer_state(tokenizer):
    fields = ("name_or_path", "vocab_size", "model_max_length", "padding_side", "truncation_side",
              "bos_token_id", "eos_token_id", "pad_token_id", "unk_token_id", "all_special_ids",
              "special_tokens_map", "added_tokens_encoder", "add_bos_token", "add_eos_token",
              "clean_up_tokenization_spaces")
    return {"class": type(tokenizer).__module__ + "." + type(tokenizer).__qualname__,
            **{key: v4_plain_state(getattr(tokenizer, key, None)) for key in fields}}


def v4_no_active_hooks(model):
    hook_points = getattr(model, "hook_dict", None)
    if not isinstance(hook_points, dict):
        raise RuntimeError("Model lacks the original TransformerLens hook registry")
    for name, point in hook_points.items():
        if getattr(point, "fwd_hooks", ()) or getattr(point, "bwd_hooks", ()):
            raise RuntimeError("Active TransformerLens hooks at " + str(name))
    if not callable(getattr(model, "modules", None)):
        raise RuntimeError("Model does not expose its module hook state")
    for module in model.modules():
        for field in ("_forward_hooks", "_forward_pre_hooks", "_backward_hooks", "_backward_pre_hooks"):
            if getattr(module, field, None):
                raise RuntimeError("Active module hooks: " + field)


def v4_parameter_state(model):
    result = []
    for parameter in model.parameters():
        result.append({"object_id": id(parameter), "version": getattr(parameter, "_version", None),
                       "shape": list(getattr(parameter, "shape", ())),
                       "dtype": str(parameter.dtype), "device": str(parameter.device),
                       "requires_grad": bool(parameter.requires_grad),
                       "data_ptr": int(parameter.data_ptr()) if callable(getattr(parameter, "data_ptr", None)) else None})
    if not result:
        raise RuntimeError("Loaded model has no parameters")
    return result


def v4_check_model(model, cfg):
    if isinstance(HookedTransformer, type) and type(model) is not HookedTransformer:
        raise RuntimeError("The loaded model is not the original HookedTransformer class")
    v4_no_active_hooks(model)
    if bool(getattr(model, "training", True)):
        raise RuntimeError("Model must already be in evaluation mode")
    if any(bool(getattr(model, key, False)) for key in ("is_loaded_in_4bit", "is_loaded_in_8bit")):
        raise RuntimeError("Quantized models cannot be reused")
    if int(model.cfg.n_ctx) != cfg["context_window"]:
        raise RuntimeError("Model context differs from the frozen backend contract")
    for field, expected in (("n_layers", 32), ("n_heads", 32), ("d_model", 4096), ("d_head", 128)):
        if getattr(model.cfg, field, None) != expected:
            raise RuntimeError("Model architecture differs at " + field)
    if getattr(model.cfg, "use_attn_result", False):
        raise RuntimeError("Model attention-result setting differs from the original z backend")
    parameters = v4_parameter_state(model)
    if any(item["dtype"] != str(torch.bfloat16) or item["requires_grad"]
           or item["device"].split(":")[0] != "cuda" for item in parameters):
        raise RuntimeError("Model parameters must all be frozen, CUDA, and bfloat16")
    if len({item["device"] for item in parameters}) != 1:
        raise RuntimeError("The original single-device model contract is required")
    tokenizer = model.tokenizer
    if not callable(getattr(tokenizer, "encode", None)) or not callable(getattr(tokenizer, "decode", None)):
        raise RuntimeError("Tokenizer encode/decode interface is missing")
    if tokenizer.eos_token_id is None or tokenizer.pad_token_id is None:
        raise RuntimeError("Tokenizer eos/padding setup is incomplete")
    module_code, modules = {}, []
    for module in model.modules():
        forward = getattr(module, "forward", None)
        function = forward.__func__ if isinstance(forward, v4_types.MethodType) else forward
        function_id = id(function)
        if function_id not in module_code:
            module_code[function_id] = (stable_hash(v4_code_state(function.__code__))
                                       if hasattr(function, "__code__") else None)
        modules.append({"object_id": id(module), "class": type(module).__module__ + "." + type(module).__qualname__,
                        "forward_object_id": function_id, "forward_code": module_code[function_id]})
    return {"parameters": parameters, "config": v4_plain_state(vars(model.cfg)),
            "tokenizer": v4_tokenizer_state(tokenizer), "modules": modules}


def v4_check_backend(backend, cfg, model_identity, expected_identity):
    if stable_hash(model_identity) != stable_hash(expected_identity):
        raise RuntimeError("Candidate model/revision/dtype/packages/Python identity differs")
    if type(backend).__name__ != "TLBackend" or stable_hash(v4_definition_state(type(backend))) != stable_hash(v4_definition_state(TLBackend)):
        raise RuntimeError("Candidate backend class differs from the copied original TLBackend")
    for name, method in vars(TLBackend).items():
        if isinstance(method, v4_types.FunctionType):
            actual = getattr(backend, name, None)
            if stable_hash(v4_definition_state(actual)) != stable_hash(v4_definition_state(method)):
                raise RuntimeError("Candidate backend method differs: " + name)
            function = actual.__func__ if isinstance(actual, v4_types.MethodType) else actual
            for dependency in ("normalize_heads", "ablation_hooks", "parse_json_output", "_json_no_duplicates"):
                reference = function.__globals__.get(dependency)
                if reference is None or stable_hash(v4_definition_state(reference)) != stable_hash(v4_definition_state(globals()[dependency])):
                    raise RuntimeError("Candidate backend dependency differs: " + dependency)
    if v4_backend_contract(backend.cfg) != v4_backend_contract(cfg) or backend.hook_kind != cfg["hook_kind"]:
        raise RuntimeError("Candidate generation settings differ")
    if backend.tokenizer is not backend.model.tokenizer:
        raise RuntimeError("Candidate backend/model tokenizers differ")
    state = v4_check_model(backend.model, cfg)
    if str(backend.device) != state["parameters"][0]["device"]:
        raise RuntimeError("Candidate backend device differs from its weights")
    return state


def v4_load_runtime(cfg, runtime_dir, package_versions, core_source, runtime_source, candidates=()):
    """Load/reuse pinned weights and return an independent parent namespace.

    The same original TLBackend wraps a compatible live model with a new config
    dictionary. No old backend configuration, run directory, or result is changed.
    Live checks protect code, identities, package versions, model/tokenizer config,
    parameter objects/versions/storage and hooks. They do not hash all 8B weight
    values; reuse trusts the supplied reviewed pinned model-run identity.
    """
    if not isinstance(cfg, dict):
        raise TypeError("cfg must be a dictionary")
    frozen_cfg = v4_copy.deepcopy(cfg)
    for key, expected in V4_FIXED_CONFIG.items():
        if key not in frozen_cfg or type(frozen_cfg[key]) is not type(expected) or frozen_cfg[key] != expected:
            raise ValueError("This notebook preserves the original fixed setting: " + key)
    if type(frozen_cfg.get("data_seed")) is not int or not 0 <= frozen_cfg["data_seed"] < 2**32:
        raise ValueError("data_seed must be an integer in [0, 2**32)")
    if not isinstance(frozen_cfg.get("experiment_version"), str) or not frozen_cfg["experiment_version"].strip():
        raise ValueError("A standalone experiment_version is required")
    packages = v4_check_packages(v4_copy.deepcopy(package_versions))
    source_states = v4_check_sources(core_source, runtime_source)
    identity = {"name": frozen_cfg["model_name"], "revision": frozen_cfg["model_revision"],
                "dtype": frozen_cfg["dtype"], "packages": packages,
                "python": v4_sys.version.split()[0]}
    implementation = stable_hash({"core_source": core_source, "runtime_source": runtime_source})
    fingerprint = stable_hash({"config": frozen_cfg, "model": identity, "implementation": implementation})
    directory = V4Path(runtime_dir).expanduser().resolve()
    manifest = {
        "schema": "conflict_v4_standalone_runtime_v1", "config": frozen_cfg,
        "model": identity, "implementation": implementation, "fingerprint": fingerprint,
        "generation_contract": v4_backend_contract(frozen_cfg),
        "sources": {"core": core_source, "runtime": runtime_source},
        "source_sha256": {"core": v4_hashlib.sha256(core_source.encode("utf-8")).hexdigest(),
                          "runtime": v4_hashlib.sha256(runtime_source.encode("utf-8")).hexdigest()},
    }
    manifest_path = directory / "manifest.json"
    if manifest_path.exists() and v4_json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
        raise RuntimeError("Locked runtime identity differs. Choose a fresh v4 directory before loading weights.")
    # This guard precedes login, downloads, rewrapping, or any inference.
    if not torch.cuda.is_available():
        raise RuntimeError("The unchanged 8B runtime requires a CUDA GPU")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("The unchanged runtime requires bfloat16 support")
    attempts, wrapped, mode, selected_label = [], None, None, None
    for index, candidate in enumerate(candidates):
        label = str(candidate.get("label", "candidate-" + str(index))) if isinstance(candidate, dict) else "candidate-" + str(index)
        try:
            if not isinstance(candidate, dict) or "backend" not in candidate or "model_identity" not in candidate:
                raise ValueError("Candidate must contain backend and model_identity")
            old_backend = candidate["backend"]
            v4_check_backend(old_backend, frozen_cfg, candidate["model_identity"], identity)
            old_cfg = v4_copy.deepcopy(old_backend.cfg)
            wrapped = TLBackend(old_backend.model, frozen_cfg)
            if old_backend.cfg != old_cfg:
                raise RuntimeError("Wrapping unexpectedly modified the previous backend config")
            v4_check_backend(wrapped, frozen_cfg, identity, identity)
            mode, selected_label = "reuse_live_model", label
            attempts.append({"label": label, "accepted": True})
            break
        except Exception as error:
            wrapped = None
            attempts.append({"label": label, "accepted": False,
                             "reason": type(error).__name__ + ": " + str(error)})
    login_called = False
    if wrapped is None:
        if not get_token():
            notebook_login()
            login_called = True
        seed_everything(frozen_cfg["data_seed"])
        model = HookedTransformer.from_pretrained(
            frozen_cfg["model_name"], revision=frozen_cfg["model_revision"],
            device="cuda", dtype=torch.bfloat16, n_ctx=frozen_cfg["context_window"],
            fold_ln=False, center_writing_weights=False, center_unembed=False,
        )
        model.eval()
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        if model.tokenizer.pad_token is None:
            model.tokenizer.pad_token = model.tokenizer.eos_token
        wrapped = TLBackend(model, frozen_cfg)
        v4_check_backend(wrapped, frozen_cfg, identity, identity)
        mode = "load_pinned_model"
    model = wrapped.model
    model_state = v4_check_model(model, frozen_cfg)
    tokenizer = model.tokenizer
    source_state_hash = stable_hash(source_states)
    manifest_snapshot = v4_copy.deepcopy(manifest)
    # Recheck before committing in case a concurrent cell changed the lock.
    if manifest_path.exists() and v4_json.loads(manifest_path.read_text(encoding="utf-8")) != manifest_snapshot:
        raise RuntimeError("Runtime manifest changed while initializing the model")
    atomic_json(manifest_path, manifest_snapshot)
    metadata = {
        "mode": mode, "reused_candidate": selected_label, "candidate_attempts": attempts,
        "login_called": login_called, "device": str(wrapped.device),
        "weight_download": "not requested" if mode == "reuse_live_model" else "HuggingFace normal local cache; download only if required",
        "model_identity_policy": "pinned loader or compatible live model with reviewed run identity",
        "live_weight_guard": "parameter identity/version/storage checks; no full weight-content digest",
        "original_backend_config_changed": False,
    }
    parent = {
        "backend": wrapped, "model": model, "RUN_DIR": directory,
        "CFG": v4_copy.deepcopy(frozen_cfg), "MODEL_IDENTITY": v4_copy.deepcopy(identity),
        "IMPLEMENTATION_HASH": implementation, "RUN_FINGERPRINT": fingerprint,
        "PACKAGE_VERSIONS": v4_copy.deepcopy(packages),
        "runtime_manifest": v4_copy.deepcopy(manifest_snapshot),
        "loader_reuse": v4_copy.deepcopy(metadata), "runtime_metadata": v4_copy.deepcopy(metadata),
    }

    def v4_verify_live_run():
        if parent.get("backend") is not wrapped or parent.get("model") is not model:
            raise RuntimeError("Standalone runtime backend/model was replaced")
        if V4Path(parent["RUN_DIR"]).expanduser().resolve() != directory:
            raise RuntimeError("Standalone runtime directory changed")
        if (stable_hash(parent["CFG"]) != stable_hash(frozen_cfg)
                or stable_hash(parent["MODEL_IDENTITY"]) != stable_hash(identity)
                or parent["IMPLEMENTATION_HASH"] != implementation
                or parent["RUN_FINGERPRINT"] != fingerprint):
            raise RuntimeError("Standalone runtime identity/config was modified")
        if wrapped.cfg != frozen_cfg:
            raise RuntimeError("Live backend configuration changed")
        if model.tokenizer is not tokenizer or wrapped.tokenizer is not tokenizer:
            raise RuntimeError("Live tokenizer object changed")
        live_states = {name: v4_definition_state(globals().get(name)) for name in source_states}
        if stable_hash(live_states) != source_state_hash:
            raise RuntimeError("Live backend/tools/runtime implementation changed")
        if v4_sys.version.split()[0] != identity["python"] or v4_check_packages(packages) != packages:
            raise RuntimeError("Live package/Python identity changed")
        v4_check_backend(wrapped, frozen_cfg, identity, identity)
        if stable_hash(v4_check_model(model, frozen_cfg)) != stable_hash(model_state):
            raise RuntimeError("Live model parameters, configuration, or tokenizer settings changed")
        if (not manifest_path.exists()
                or v4_json.loads(manifest_path.read_text(encoding="utf-8")) != manifest_snapshot):
            raise RuntimeError("Standalone runtime manifest changed on disk")
        if stable_hash(parent["runtime_manifest"]) != stable_hash(manifest_snapshot):
            raise RuntimeError("In-memory runtime manifest changed")
        return True

    parent["verify_live_run"] = v4_verify_live_run
    v4_verify_live_run()
    reuse_path = directory / "loader-reuse.json"
    previous = v4_json.loads(reuse_path.read_text(encoding="utf-8")) if reuse_path.exists() else {"run_fingerprint": fingerprint, "loads": []}
    if previous.get("run_fingerprint") != fingerprint or not isinstance(previous.get("loads"), list):
        raise RuntimeError("Loader reuse provenance belongs to another runtime")
    previous["loads"].append(metadata)
    atomic_json(reuse_path, previous)
    return parent
'''
V6_RUNTIME_NAMESPACE = {'__name__': 'v6_runtime', 'np': np, 'torch': torch,
                        'HookedTransformer': HookedTransformer, 'get_token': get_token, 'notebook_login': notebook_login}
for v5_source, v5_filename in ((V6_CORE_SOURCE, '<v5-core>'), (V6_RUNTIME_SOURCE, '<v5-runtime>')):
    linecache.cache[v5_filename] = (len(v5_source), None, v5_source.splitlines(True), v5_filename)
    exec(compile(v5_source, v5_filename, 'exec'), V6_RUNTIME_NAMESPACE)


In [6]:
V6_LEGACY_SOURCE = r'''from functools import lru_cache
import copy as gx_copy

GX_ATTRIBUTES = ("delivery_city", "order_customer", "preferred_item")

GX_DEMO_CITIES = ("Valletta", "Reykjavik", "Suva", "La Paz")

GX_ORDER_FIELDS = {
    "delivery_city": "city", "order_customer": "customer", "preferred_item": "item",
}

def gx_task_payload(case):
    """Only the other two fixed fields and fallback accompany the request ID."""
    return {"request_id": case["request_id"], "resolve_attribute": case["attribute"],
            "fixed_fields": {attribute: case["gold_fields"][attribute]
                             for attribute in GX_ATTRIBUTES if attribute != case["attribute"]},
            "fallback_item": case["fallback_item"]}

def gx_actor_examples():
    """Independent tool examples cover all field mappings and real fallback.

    These short teaching cases use only demo cities and identifiers. They never
    depend on evaluation cases, and do not furnish actions for evaluated tasks.
    """
    examples = []
    for index, attribute in enumerate(GX_ATTRIBUTES):
        request = ("req-demo-71d3", "req-demo-a926", "req-demo-4f80")[index]
        customer = ("acct-demo-8c31", "acct-demo-90ea", "acct-demo-2d74")[index]
        preferred = ("sku-demo-31ab", "sku-demo-e072", "sku-demo-64dc")[index]
        fallback = ("sku-demo-8f62", "sku-demo-72de", "sku-demo-0a13")[index]
        city = GX_DEMO_CITIES[index]
        fields = {"delivery_city": city, "order_customer": customer, "preferred_item": preferred}
        available = index != 2
        env = OrderEnvironment({"stock": {preferred: int(available), fallback: 2},
                                "next_reservation": (7, 19, 41)[index]})
        events = []

        def gx_demo_step(action):
            observation = env.step(action)
            events.append({"action": action, "observation": observation})
            return observation

        observation = gx_demo_step({"tool": "check_stock", "item": preferred})
        chosen = preferred if observation["available"] > 0 else fallback
        if chosen != preferred:
            gx_demo_step({"tool": "check_stock", "item": fallback})
        reservation = gx_demo_step({"tool": "reserve_item", "item": chosen})["reservation"]
        gx_demo_step({"tool": "create_order", "customer": customer, "item": chosen,
                      "city": city, "reservation": reservation})
        examples.append({
            "task": {"request_id": request, "resolve_attribute": attribute,
                     "fixed_fields": {key: value for key, value in fields.items() if key != attribute},
                     "fallback_item": fallback},
            "memory": [{"kind": "verified_profile", "entity": request, "attribute": attribute,
                        "value": fields[attribute], "source_event": "evt-demo-" + str(index)}],
            "events": events,
        })
    return examples

@lru_cache(maxsize=1)
def gx_actor_example_text():
    """Teach every next-call phase, including the initially empty history.

    The v5 smoke model copied the final-call pattern before any reservation.
    Render every legal prefix of the same three independent teaching cases,
    not only the final create_order prefix. Every observation is produced by
    the unchanged tools. All ten examples are static and identical across arms;
    no demonstration is chosen using an evaluated task or its hidden truth.
    """
    continuations = []
    for example in gx_actor_examples():
        for step, event in enumerate(example["events"]):
            continuations.append(
                "INDEPENDENT EXAMPLE:\nTASK_JSON:\n" + canonical_json(example["task"])
                + "\nMEMORY_JSON:\n" + canonical_json(example["memory"])
                + "\nEXECUTED_TOOL_HISTORY_JSON:\n" + canonical_json(example["events"][:step])
                + "\nNEXT_TOOL_CALL:\n" + canonical_json(event["action"])
                + "\nEND_EXAMPLE")
    return "\n".join(continuations)

def actor_prompt(case, memory, events):
    """Present the effective arm's memory verbatim; do not insert focal gold."""
    visible = [{"action": event.get("action"), "observation": event.get("observation")}
               for event in events if event.get("action") is not None]
    return (
        "You are an order-processing assistant. Execute one order using the tools.\n"
        "A request_id identifies a request, not a customer account. Resolve the indicated attribute "
        "for that request from its records, then combine it with the task's fixed_fields. "
        "delivery_city maps to order.city; order_customer maps to order.customer; preferred_item "
        "is the SKU to check and reserve when available. The fallback_item is allowed only if "
        "the resolved preferred SKU is out of stock.\n"
        "The records belong to one system. Match the request ID and the indicated attribute. Actions and tool observations do not update the request fields.\n"
        "Check the preferred SKU's stock. If unavailable, check the fallback. Reserve the chosen "
        "available SKU, then create exactly one order. Copy the reservation ID only from its "
        "successful reserve_item observation; a stock count is not a reservation.\n"
        "Output exactly one complete JSON object for the next tool call, without explanation. "
        "Every create_order requires all four arguments: customer, item, city, reservation. "
        "Use city as the tool key even when the task field is named delivery_city.\n"
        'Tools: {"tool":"check_stock","item":"SKU"}; '
        '{"tool":"reserve_item","item":"SKU"}; '
        '{"tool":"create_order","customer":"ACCOUNT","item":"SKU","city":"CITY","reservation":"RESERVATION"}.\n'
        "Independent next-call examples (never copy their values or IDs):\n"
        + gx_actor_example_text()
        + "\nEND_EXAMPLES. Execute this independent request:\n"
        + case["template_text"].format(request_id=case["request_id"])
        + "\nOnly this request's EXECUTED_TOOL_HISTORY_JSON records completed actions. "
        "An empty history means nothing has been checked or reserved: start with check_stock "
        "for the resolved preferred SKU. After a positive stock result use reserve_item. "
        "If the preferred SKU is unavailable, check the fallback before reserving it. "
        "Use create_order only after reserve_item succeeded in this request's history. "
        "Never invent a reservation ID or take one from an example."
        + "\nGenerate only the next call. For create_order include customer, item, city, "
        "and the reservation from a successful reserve_item observation. Resolve city from "
        "this request's delivery_city, including when it is supplied in fixed_fields."
        + "\nTASK_JSON:\n" + canonical_json(gx_task_payload(case))
        + "\nMEMORY_JSON:\n" + canonical_json(memory)
        + "\nEXECUTED_TOOL_HISTORY_JSON:\n" + canonical_json(visible)
        + "\nNEXT_TOOL_CALL:\n"
    )

def gx_expected_order(case, initial_state=None):
    """Evaluator-only final fields, including stock-contingent SKU truth."""
    state = case["initial_env"] if initial_state is None else initial_state
    fields = case["gold_fields"]
    preferred = fields["preferred_item"]
    item = preferred if state["stock"].get(preferred, 0) > 0 else case["fallback_item"]
    return {"city": fields["delivery_city"], "customer": fields["order_customer"], "item": item}

def gx_value_missing(value):
    """Absent/null/blank values are not explicit choices of a wrong value.

    Nonstring values are schema/type failures, not choices of a string-valued
    attribute. They are excluded from wrong-choice rates by gx_wrong_value;
    the schema-complete indicators still expose their invalidity.
    """
    return value is None or (isinstance(value, str) and not value.strip())

def gx_wrong_value(value, expected):
    """Count only an explicitly supplied, nonblank string that is incorrect."""
    return isinstance(value, str) and bool(value.strip()) and value != expected

def gx_order_schema_complete(action):
    """A complete order attempt has exactly five keys with nonblank strings.

    This is a diagnostic; the original executable schema remains unchanged.
    It deliberately treats a whitespace-only value as missing even if the
    executable tool's nonempty-string check would accept that value.
    """
    return (isinstance(action, dict)
            and set(action) == {"tool", "customer", "item", "city", "reservation"}
            and action.get("tool") == "create_order"
            and all(isinstance(value, str) and bool(value.strip()) for value in action.values()))

def gx_score_actor(case, events, orders, initial_state):
    """Score accepted fields and the executed prerequisite subsequence.

    A wrong focal field is counted on any accepted order, independently of the
    correctness of the other fields. Missing orders are failures and have their
    own indicator; they never masquerade as correct field decisions. Tool
    recovery is allowed: strict success requires a valid final check/reserve/
    order subsequence, while all earlier tool errors remain reported separately.

    Intermediate rates are source-level booleans, not counts of repeated calls.
    The first order attempt is the first parsed create_order action, including
    a rejected one; any-attempt metrics cover all such actions. Missing or
    nonstring focal values do not count as explicit wrong choices. Reservation
    acceptance requires that reserve_item's own observation has ok=True.
    Expected items include the legal initial-stock fallback. Conflict-SKU
    reservation metrics apply only to preferred_item sources (None otherwise).
    """
    expected = gx_expected_order(case, initial_state)
    focal = GX_ORDER_FIELDS[case["attribute"]]
    correct = len(orders) == 1 and all(orders[0].get(key) == value for key, value in expected.items())
    preferred = case["gold_fields"]["preferred_item"]
    chosen = expected["item"]
    checked, grounded = {}, set()
    checked_preferred = checked_chosen = reserved_chosen = path_valid = False
    tool_errors = []
    actions = []
    order_attempts, reservation_attempts, accepted_reservations = [], [], []
    for step, event in enumerate(events):
        action = event.get("action")
        observation = event.get("observation")
        if isinstance(action, dict):
            actions.append(action)
            if action.get("tool") == "create_order":
                order_attempts.append(action)
            elif action.get("tool") == "reserve_item":
                reservation_attempts.append(action)
                if isinstance(observation, dict) and observation.get("ok") is True:
                    accepted_reservations.append(action)
        if not isinstance(action, dict) or not isinstance(observation, dict):
            continue
        if not observation.get("ok", False):
            tool_errors.append({"step": step, "error": observation.get("error")})
            continue
        tool, item = action.get("tool"), action.get("item")
        if tool == "check_stock":
            checked[item] = observation["available"]
            checked_preferred |= item == preferred
            checked_chosen |= item == chosen and observation["available"] > 0
        elif tool == "reserve_item" and item == chosen:
            reserved_chosen = True
            priority_ok = (checked.get(preferred, -1) > 0 if chosen == preferred
                           else checked.get(preferred, -1) == 0)
            if priority_ok and checked.get(chosen, 0) > 0:
                grounded.add(observation["reservation"])
        elif tool == "create_order" and item == chosen:
            path_valid |= action.get("reservation") in grounded
    protocol_errors = {"action_must_be_object", "unknown_tool", "invalid_arguments"}
    format_failure = any(event.get("parse_error") is not None for event in events) or any(
        error["error"] in protocol_errors for error in tool_errors)
    first_order = order_attempts[0] if order_attempts else None
    order_fields = ("customer", "item", "city", "reservation")
    first_missing_attribute = first_order is not None and gx_value_missing(first_order.get(focal))
    first_missing_city = first_order is not None and gx_value_missing(first_order.get("city"))
    first_missing_required = first_order is not None and any(
        gx_value_missing(first_order.get(field)) for field in order_fields)
    conflict_item_applicable = case["attribute"] == "preferred_item"
    return {
        "task_success": bool(correct), "strict_task_success": bool(correct and path_valid),
        "accepted_wrong_attribute": any(order.get(focal) != expected[focal] for order in orders),
        "accepted_conflict_value": any(order.get(focal) == case["conflict_value"] for order in orders),
        "accepted_correct_attribute": any(order.get(focal) == expected[focal] for order in orders),
        "accepted_wrong_other_fields": any(any(order.get(key) != value for key, value in expected.items()
                                                if key != focal) for order in orders),
        "no_accepted_order": not orders, "accepted_order_count": len(orders),
        "format_failure": bool(format_failure), "tool_error": bool(tool_errors),
        "tool_error_count": len(tool_errors), "action_count": len(actions),
        "order_attempted": first_order is not None,
        "order_accepted": bool(orders), "order_attempt_count": len(order_attempts),
        "first_order_attribute_correct": first_order is not None and first_order.get(focal) == expected[focal],
        "first_order_conflict_value": first_order is not None and first_order.get(focal) == case["conflict_value"],
        "any_order_conflict_value": any(action.get(focal) == case["conflict_value"] for action in order_attempts),
        "first_order_wrong_attribute": first_order is not None and gx_wrong_value(first_order.get(focal), expected[focal]),
        "any_order_wrong_attribute": any(gx_wrong_value(action.get(focal), expected[focal]) for action in order_attempts),
        "first_order_missing_attribute": bool(first_missing_attribute),
        "any_order_missing_attribute": any(gx_value_missing(action.get(focal)) for action in order_attempts),
        "first_order_missing_city": bool(first_missing_city),
        "any_order_missing_city": any(gx_value_missing(action.get("city")) for action in order_attempts),
        "first_order_missing_required_fields": bool(first_missing_required),
        "any_order_missing_required_fields": any(any(gx_value_missing(action.get(field)) for field in order_fields)
                                                  for action in order_attempts),
        "first_order_schema_complete": first_order is not None and gx_order_schema_complete(first_order),
        "any_order_schema_complete": any(gx_order_schema_complete(action) for action in order_attempts),
        "reservation_attempted": bool(reservation_attempts),
        "reservation_accepted": bool(accepted_reservations),
        "reservation_attempt_count": len(reservation_attempts),
        "accepted_reservation_count": len(accepted_reservations),
        "attempted_wrong_reservation_item": any(gx_wrong_value(action.get("item"), chosen)
                                                 for action in reservation_attempts),
        "accepted_wrong_reservation_item": any(gx_wrong_value(action.get("item"), chosen)
                                                for action in accepted_reservations),
        "attempted_conflict_reservation_item": (
            any(action.get("item") == case["conflict_value"] for action in reservation_attempts)
            if conflict_item_applicable else None),
        "accepted_conflict_reservation_item": (
            any(action.get("item") == case["conflict_value"] for action in accepted_reservations)
            if conflict_item_applicable else None),
        "prerequisite_path_valid": bool(path_valid),
        "checked_preferred": bool(checked_preferred), "checked_chosen_available": bool(checked_chosen),
        "reserved_chosen": bool(reserved_chosen),
        "subgoal_progress": (int(checked_preferred) + int(checked_chosen) + int(reserved_chosen) + int(correct)) / 4,
    }
'''


In [7]:
V6_TASKS_SOURCE = r'''"""Truthful request pairs, matched contexts, and the original three-tool workflow."""
import copy
import json
import random
from collections import Counter
from functools import lru_cache


TEMPLATES = {
    "smoke": ("Process request {request_id}. Place exactly one order using its resolved fields.",),
    "pilot": ("Fulfil request {request_id} by placing one order using its resolved fields.",
              "Arrange one shipment for request {request_id} using its resolved order details."),
    "confirm": ("Submit one order for request {request_id} using the fields belonging to that request.",
                "Complete the purchase specified by request {request_id} using its recorded details."),
}
SPLIT_RANGES = {"smoke": (100000, 199999), "pilot": (200000, 399999), "confirm": (400000, 999999)}
DATA_SEED = 961103


def make_worlds(split, count):
    if split not in SPLIT_RANGES or type(count) is not int or not 0 < count <= 10000:
        raise ValueError("Unknown split or invalid number of independent worlds")
    rng = random.Random(f"v6/worlds/{split}/{DATA_SEED}")
    lo, hi = SPLIT_RANGES[split]
    used = set()

    def identifier(kind):
        while True:
            digits = str(rng.randint(lo, hi))
            if digits not in used:
                used.add(digits)
                return f"{kind}-{digits[:3]}-{digits[3:]}"

    worlds = []
    for i in range(count):
        # The first n worlds are stable if a larger manifest is generated later.
        # A/B, gold, and competitor appear only in evaluator metadata.
        requests = [identifier("req"), identifier("req")]
        items = [identifier("sku"), identifier("sku")]
        lots = [identifier("lot"), identifier("lot")]
        worlds.append({
            "id": f"{split}-{i+1:05d}", "index": i, "split": split,
            "requests": requests, "items": items, "lots": lots,
            "events": [identifier("evt"), identifier("evt")],
            "fallback": identifier("sku"), "customer": identifier("acct"),
            "city": rng.choice(("Lisbon", "Osaka", "Cairo", "Prague", "Mumbai", "Nairobi")),
            "template": i % len(TEMPLATES[split]), "reverse": bool((i // 2) % 2),
        })
    return worlds


def case_for(w, target, stocked=True):
    other = 1 - target
    stock = {sku: 2 for sku in (*w["items"], w["fallback"])}
    stock[w["items"][target]] = 2 if stocked else 0
    return {
        "source_id": w["id"], "stage": w["split"], "attribute": "preferred_item",
        "request_id": w["requests"][target], "template_id": str(w["template"]),
        "template_text": TEMPLATES[w["split"]][w["template"]],
        "gold_fields": {"preferred_item": w["items"][target], "delivery_city": w["city"],
                        "order_customer": w["customer"]},
        "conflict_value": w["items"][other], "fallback_item": w["fallback"],
        "profile_event_id": w["events"][target], "note_event_id": w["events"][other],
        "stock_available": stocked, "initial_env": {"stock": stock, "orders": [], "reservations": {},
                                                   "next_reservation": 1, "next_order": 1},
    }


def records_for(w, target, context):
    if context not in ("C0", "C1") or target not in (0, 1):
        raise ValueError("Invalid target/context")
    records = []
    for index in range(2):
        has_item = index == target or context == "C1"
        records.append({"kind": "verified_profile", "entity": w["requests"][index],
                        "attribute": "preferred_item" if has_item else "dispatch_batch",
                        "value": w["items"][index] if has_item else w["lots"][index],
                        "source_event": w["events"][index]})
    return list(reversed(records)) if w["reverse"] else records


@lru_cache(maxsize=1)
def direct_examples():
    lines = []
    for w in make_worlds("smoke", 3):
        # Distinct demo-only identifiers even from the usability smoke.
        for group in ("requests", "items", "lots", "events"):
            w[group] = [v.replace("-", "-demo-", 1) for v in w[group]]
        for t in (0, 1):
            lines.append("EXAMPLE REQUEST " + w["requests"][t] + "\nRECORDS " +
                         canonical_json(records_for(w, t, ("C0", "C1")[t])) +
                         "\nANSWER " + canonical_json({"item": w["items"][t]}))
    return "\n".join(lines)


def direct_prompt(w, target, context):
    # Same target, fixed fields, records and order as Actor. Read-only, one model call.
    case = case_for(w, target)
    return ("Read the request records. Answer with one JSON object containing only the item field. "
            "Return the preferred SKU for the specified request.\n" + direct_examples() +
            "\nTASK_JSON:\n" + canonical_json(gx_task_payload(case)) +
            "\nMEMORY_JSON:\n" + canonical_json(records_for(w, target, context)) + "\nANSWER:\n")


def arm_sets(global_heads, random_count):
    global_heads = [tuple(h) for h in global_heads]
    if (len(global_heads) != 80 or len(set(global_heads)) != 80
            or any(not (0 <= l < 32 and 0 <= h < 32) for l, h in global_heads)):
        raise ValueError("Expected 80 distinct query heads of the original Llama-3.1-8B")
    arms = {"none": [], "global80": global_heads}
    for draw in range(random_count):
        rng = random.Random(61401 + draw)
        pick = []
        for layer, count in sorted(Counter(l for l, h in global_heads).items()):
            pool = [h for h in range(32) if (layer, h) not in global_heads]
            pick.extend((layer, h) for h in rng.sample(pool, count))
        arms[f"random{draw+1}"] = sorted(pick)
    return arms


def run_actor(backend, case, records, heads, forced_check=False):
    env = OrderEnvironment(case["initial_env"])
    events = []
    if forced_check:
        action = {"tool": "check_stock", "item": case["gold_fields"]["preferred_item"]}
        events.append({"action": action, "observation": env.step(action), "parse_error": None,
                       "origin": "fixed_prefix", "step": 0})
    end = "action_budget"
    while len(events) < backend.cfg["max_actions"]:
        prompt = actor_prompt(case, records, events)
        if len(backend.encode(prompt)) > backend.cfg["max_prompt_tokens"]:
            # A pathological generated argument can be echoed in multiple tool errors.
            # Score this execution failure rather than truncate evidence or lose a world.
            end = "prompt_budget"
            break
        out = backend.generate(prompt, role="actor", heads=heads, use_cache=True)
        action, error = parse_json_output(out["text"], dict)
        obs = None if error else env.step(action)
        events.append({"step": len(events), "origin": "model", "prompt": prompt,
                       "action": action, "observation": obs, "parse_error": error, "generation": out})
        if error:
            end = "format_failure"
            break
        if obs.get("error") in ("action_must_be_object", "unknown_tool", "invalid_arguments"):
            end = "protocol_failure"
            break
        if action.get("tool") == "create_order" and obs.get("ok"):
            end = "order_created"
            break
    orders = env.state["orders"]
    metrics = gx_score_actor(case, events, orders, case["initial_env"])
    first = (events[0]["action"] or {}) if events else {}
    first_ok = bool(events and events[0]["observation"] and events[0]["observation"].get("ok"))
    checked = first_ok and first.get("tool") == "check_stock"
    metrics.update(first_lookup_other=bool(checked and first.get("item") == case["conflict_value"]),
                   first_lookup_gold=bool(checked and first.get("item") == case["gold_fields"]["preferred_item"]),
                   first_valid_lookup=bool(checked), prompt_budget_hit=end=="prompt_budget",
                   action_budget_hit=end=="action_budget")
    return {"metrics": metrics, "events": events, "initial_state": case["initial_env"],
            "final_state": env.snapshot(), "terminal": end, "forced_check": forced_check}


def direct_choice(backend, w, target, context):
    prompt = direct_prompt(w, target, context)
    out = backend.generate(prompt, role="actor", heads=(), max_new_tokens=48, use_cache=True)
    answer, error = parse_json_output(out["text"], dict)
    schema_ok = not error and set(answer) == {"item"} and isinstance(answer["item"], str) and bool(answer["item"].strip())
    item = answer["item"] if schema_ok else None
    valid = schema_ok and item in (*w["items"],w["fallback"])
    return {"prompt": prompt, "generation": out, "item": item, "valid": bool(valid), "schema_valid": bool(schema_ok),
            "parse_error": error, "other": item == w["items"][1-target], "correct": item == w["items"][target]}


def candidate_tokens(backend, prefix, item):
    # Every candidate begins with sku-. The identical stem cancels from the log odds.
    if not item.startswith("sku-"):
        raise ValueError("Expected a SKU")
    ids = backend.encode(prefix)
    full = backend.encode(prefix + item[4:] + '"}')
    if full[:len(ids)] != ids or not len(ids) < len(full) <= backend.cfg["max_prompt_tokens"]:
        raise ValueError("Candidate retokenized the controlled prefix, or exceeded the context budget")
    return ids, full


def score_pair(backend, prompt, gold, other, heads):
    prefix = prompt + '{"tool":"check_stock","item":"sku-'
    alternatives = [candidate_tokens(backend, prefix, v) for v in (gold, other)]
    if len(alternatives[0][1]) != len(alternatives[1][1]):
        raise ValueError("The two complete value/terminator sequences must have equal token counts")
    scores = []
    with torch.inference_mode(), backend.model.hooks(fwd_hooks=ablation_hooks(backend.model, heads)):
        for before, full in alternatives:
            tokens = torch.tensor([full[:-1]], dtype=torch.long, device=backend.device)
            # Each row predicts the NEXT token. Include the identical JSON terminator.
            logits = backend.model(tokens, return_type="logits")[0, len(before)-1:, :].float()
            targets = torch.tensor(full[len(before):], device=backend.device)
            selected = logits.log_softmax(-1).gather(1, targets[:, None])
            scores.append(float(selected.sum().item()))
    return {"gold_logp": scores[0], "other_logp": scores[1], "margin": scores[0]-scores[1],
            "candidate_tokens": len(alternatives[0][1])-len(alternatives[0][0])}


def check_inputs(backend, worlds):
    maximum = 0
    for w in worlds:
        for target in (0, 1):
            case = case_for(w, target)
            lengths = []
            for context in ("C0", "C1"):
                records = records_for(w, target, context)
                prompt = actor_prompt(case, records, [])
                dp = direct_prompt(w, target, context)
                prefix = prompt + '{"tool":"check_stock","item":"sku-'
                counts = [len(candidate_tokens(backend, prefix, item)[1]) for item in w["items"]]
                if counts[0] != counts[1]:
                    raise ValueError(f"SKU tokenizer mismatch in {w['id']}")
                lengths.append((len(backend.encode(prompt)), len(backend.encode(dp))))
                maximum = max(maximum, *counts, *lengths[-1])
            if lengths[0] != lengths[1]:
                raise ValueError(f"C0/C1 token counts differ in {w['id']}, target {target}: {lengths}")
    if maximum > backend.cfg["max_prompt_tokens"]:
        raise ValueError("Untruncated prompt exceeds budget")
    return {"worlds": len(worlds), "equal_lengths": True, "max_initial_tokens": maximum,
            "later_steps": "original backend checks each full prompt before generation"}
'''


In [8]:
V6_STUDY_SOURCE = r'''"""One atomic checkpoint per condition/target/arm, plus separate forced-history controls."""
import csv
import hashlib
import math
import os
import tempfile
import time
import zipfile
from pathlib import Path


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(dir=path.parent, prefix=".pending-", suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(value, f, ensure_ascii=False, allow_nan=False, separators=(",", ":"))
        os.replace(temporary, path)
    finally:
        Path(temporary).unlink(missing_ok=True)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(dict.fromkeys(k for row in rows for k in row))
    fd, temporary = tempfile.mkstemp(dir=path.parent, prefix=".pending-", suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            writer.writerows(rows)
        os.replace(temporary, path)
    finally:
        Path(temporary).unlink(missing_ok=True)


def paper_reference(backend, paper, arms):
    def margin(prompt, gold, other, heads):
        with torch.inference_mode(), backend.model.hooks(fwd_hooks=ablation_hooks(backend.model, heads)):
            logits = backend.model(backend.tokens(prompt), return_type="logits")[0, -1]
            return float((logits[gold].float() - logits[other].float()).item())
    rows = []
    for c in paper["cases"]:
        gold, other = c["gold_token_id"], c["distractor_token_id"]
        for name, tok in (("gold", gold), ("distractor", other)):
            if backend.tokenizer.encode(c[name+"_token_str"], add_special_tokens=False) != [tok]:
                raise ValueError("Paper reference token ID mismatch")
        row = dict(c)
        row["clean_none"] = margin(c["clean_prompt"], gold, other, [])
        for arm in ("none", "global80", "random1"):
            row["attacked_"+arm] = margin(c["attacked_prompt"], gold, other, arms[arm])
        rows.append(row)
    drop = np.mean([r["clean_none"]-r["attacked_none"] for r in rows])
    gain = np.mean([r["attacked_global80"]-r["attacked_none"] for r in rows])
    rand = np.mean([r["attacked_random1"]-r["attacked_none"] for r in rows])
    return {"rows": rows, "mean_attack_drop": float(drop), "global80_gain": float(gain),
            "random1_gain": float(rand), "direction_pass": bool(drop > 0 and gain > 0)}


class Study:
    def __init__(self, root, backend, worlds, smoke_worlds, arms, identity, verify, scorer=score_pair):
        self.root, self.backend = Path(root), backend
        self.worlds, self.smoke_worlds = worlds, smoke_worlds
        self.arms, self.verify, self.scorer = arms, verify, scorer
        self.manifest = dict(identity, worlds=worlds, smoke_worlds=smoke_worlds, arms=arms)
        self.signature = stable_hash(self.manifest)
        lock = self.root / "manifest.json"
        if lock.exists():
            if stable_hash(read_json(lock)) != self.signature:
                raise RuntimeError(" Runs different identities or samples. Please select a new one.  RUN_LABEL， Keeps the original directory. ")
        else:
            if self.root.exists() and any(self.root.iterdir()):
                raise RuntimeError(" Turns out the directory is not empty but missing  manifest， Please keep the original directory and select a new label. ")
            save_json(lock, self.manifest)
        self.new_units = 0

    def check(self):
        self.verify()
        if stable_hash(self.manifest) != self.signature or stable_hash(read_json(self.root/"manifest.json")) != self.signature:
            raise RuntimeError(" Run configuration or disk  manifest  Change ")

    def unit(self, key, fn=None):
        path = self.root / (key + ".json")
        tag = stable_hash([self.signature, key])
        if path.exists():
            obj = read_json(path)
            if obj["signature"] != tag or obj["digest"] != stable_hash(obj["result"]):
                raise RuntimeError(" Checkpoint check failed : " + str(path))
            return obj["result"]
        if fn is None:
            return None
        value = fn()
        save_json(path, {"signature": tag, "digest": stable_hash(value), "result": value})
        self.new_units += 1
        return value

    def smoke(self):
        self.check()
        rows = []
        for w in self.smoke_worlds:
            t = w["index"] % 2
            def execute():
                case = case_for(w, t, stocked=w["index"] % 2 == 0)
                act = run_actor(self.backend, case, records_for(w, t, "C0"), [])
                return {"actor": act, "direct": direct_choice(self.backend, w, t, "C0")}
            out = self.unit("smoke/"+w["id"], execute)
            m = out["actor"]["metrics"]
            rows.append({"world": w["id"], "stocked": w["index"] % 2 == 0,
                         "strict": m["strict_task_success"], "first_valid": m["first_lookup_gold"],
                         "format_failure": m["format_failure"], "direct_correct": out["direct"]["correct"]})
        ok = (sum(r["strict"] for r in rows) >= 5 and all(r["first_valid"] for r in rows)
              and not any(r["format_failure"] for r in rows) and sum(r["direct_correct"] for r in rows) >= 5
              and all(sum(r["strict"] for r in rows if r["stocked"] == stock) >= 2 for stock in (True, False)))
        result = {"pass": bool(ok), "rows": rows}
        if self.unit("controls/readiness",lambda: result) != result:
            raise RuntimeError("Readiness summary disagrees with saved smoke trajectories")
        save_json(self.root/"smoke-report.json", result)
        return result

    def prepare(self, paper):
        self.check()
        audit = self.unit("controls/input-audit", lambda: check_inputs(self.backend, self.worlds+self.smoke_worlds))
        reference = self.unit("controls/old-paper", lambda: paper_reference(self.backend, paper, self.arms))
        smoke = self.smoke()
        print(" Enter pair check ", audit)
        print(" The original paper is in contrast. ", {k:v for k,v in reference.items() if k != "rows"})
        print(" Tools and a reply pre-check ", smoke)
        return smoke, reference

    def main_unit(self, w, t, context, arm):
        case, records, heads = case_for(w, t), records_for(w, t, context), self.arms[arm]
        result = {"world": w["id"], "target": t, "context": context, "arm": arm,
                  "template": w["template"], "records": records, "target_id": case["request_id"]}
        result["probe"] = self.scorer(self.backend, actor_prompt(case, records, []),
                                      case["gold_fields"]["preferred_item"], case["conflict_value"], heads)
        result["actor"] = run_actor(self.backend, case, records, heads)
        if arm == "none":
            result["direct"] = direct_choice(self.backend, w, t, context)
        return result

    def run(self, arm_names, limit=None):
        self.check()
        if limit is not None and (type(limit) is not int or limit < 1):
            raise ValueError("WORLDS_PER_CALL must be a positive integer or None")
        readiness = self.unit("controls/readiness")
        if not readiness or not readiness["pass"] or self.unit("controls/input-audit") is None:
            raise RuntimeError(" First complete and pass six independent pre-screens, check.  smoke-report.json  And the original output. ")
        if any(a != "none" for a in arm_names) and not self.unit("controls/old-paper")["direction_pass"]:
            raise RuntimeError(" The original-paper positive control did not recover the expected direction. Intact results were retained; inspect the model and masks first. ")
        for arm in arm_names:
            done = 0
            for w in self.worlds:
                keys = [(t,c,f"main/{w['id']}/{arm}-{t}-{c}") for t in (0,1) for c in ("C0","C1")]
                # First eight prespecified worlds; one balanced target; identical prefix except stock observation.
                forks = [(stock,f"fork/{w['id']}/{arm}-{int(stock)}") for stock in (True,False)] if w["index"] < 8 else []
                if all((self.root/(k+".json")).exists() for *_,k in [*keys,*forks]):
                    continue
                for t,c,k in keys:
                    self.unit(k, lambda: self.main_unit(w, t, c, arm))
                for stock,k in forks:
                    t = w["index"] % 2
                    self.unit(k, lambda: {"world":w["id"], "target":t, "arm":arm, "stocked":stock,
                        "actor":run_actor(self.backend, case_for(w,t,stock), records_for(w,t,"C1"),
                                          self.arms[arm], forced_check=True)})
                done += 1
                print(f"[{arm}] {w['id']} complete | {self.new_units} new checkpoint units", flush=True)
                if limit is not None and done >= limit:
                    break
        self.check()

    def collect(self):
        main, forks = [], []
        for w in self.worlds:
            for arm in self.arms:
                for t in (0,1):
                    for ctx in ("C0","C1"):
                        r = self.unit(f"main/{w['id']}/{arm}-{t}-{ctx}")
                        if r is None:
                            continue
                        m = r["actor"]["metrics"]
                        calls = [e["generation"] for e in r["actor"]["events"] if e["origin"]=="model"]
                        row = {k:r[k] for k in ("world","target","context","arm","template")}
                        row.update(m, **r["probe"], model_calls=len(calls), output_tokens=sum(c["token_count"] for c in calls),
                                   generation_seconds=sum(c["seconds"] for c in calls))
                        if "direct" in r:
                            row.update({"direct_"+k:r["direct"][k] for k in ("valid","schema_valid","correct","other")})
                        main.append(row)
                if w["index"] < 8:
                    for stock in (True,False):
                        r = self.unit(f"fork/{w['id']}/{arm}-{int(stock)}")
                        if r is not None:
                            forks.append({"world":w["id"], "target":r["target"], "arm":arm,
                                          "stocked":stock, **r["actor"]["metrics"]})
        return main, forks

    def export(self):
        main, forks = self.collect()
        report = analyze(self, main, forks)
        dest = self.root/"reports"
        write_csv(dest/"main.csv", main)
        write_csv(dest/"stock-forks.csv", forks)
        write_csv(dest/"conditions.csv", report["conditions"])
        write_csv(dest/"paired-effects.csv", report["effects"])
        write_csv(dest/"power-planning.csv", report["planning"])
        save_json(dest/"report.json", report)
        archive = self.root.parent/("agent_entrainment_v6_"+self.root.parent.name+"_"+report["status"]+"_results.zip")
        tmp = archive.with_suffix(".zip.pending")
        with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as z:
            for f in sorted(self.root.rglob("*")):
                if f.is_file() and f.suffix in (".json",".csv"):
                    z.write(f, f.relative_to(self.root))
        os.replace(tmp,archive)
        return report, archive
'''


In [9]:
V6_REPORTS_SOURCE = r'''"""Paired, world-level pilot estimation. A and B never count as independent worlds."""
import math
from functools import lru_cache
from statistics import NormalDist
import numpy as np


def effect_row(name, values, unit):
    a = np.asarray(values, dtype=float)
    out = {"contrast": name, "unit": unit, "n_worlds": len(a),
           "estimate": None, "ci95_low": None, "ci95_high": None,
           "positive_worlds": int((a>0).sum()), "negative_worlds": int((a<0).sum()),
           "zero_worlds": int((a==0).sum())}
    if len(a):
        out["estimate"] = float(a.mean())
    if len(a) > 1 and np.ptp(a) > 0:
        rng = np.random.default_rng(76102)
        draws = np.concatenate([a[rng.integers(len(a),size=(100,len(a)))].mean(axis=1) for _ in range(30)])
        lo, hi = np.quantile(draws,[.025,.975])
        out.update(ci95_low=float(lo), ci95_high=float(hi))
    return out


@lru_cache(maxsize=512)
def exact_scenario_power(n, q, delta):
    """Unconditional power of the two-sided exact conditional McNemar test.

    One predesignated binary observation per world in each of two conditions.
    q = P(10)+P(01); delta = P(01)-P(10). Normality is not assumed.
    """
    from scipy.stats import binom
    if not (n > 0 and 0 < q < 1 and 0 < delta <= q):
        raise ValueError("Require 0 < delta <= discordance < 1")
    lo, hi = binom.ppf([1e-13,1-1e-13], n, q).astype(int)
    d = np.arange(lo,hi+1)
    k = binom.ppf(.025,d,.5).astype(int)
    k -= (binom.cdf(k,d,.5) > .025 + 1e-14).astype(int)
    p = (1 + delta/q)/2
    reject = binom.cdf(k,d,p) + binom.sf(d-k-1,d,p)
    return float(np.sum(binom.pmf(d,n,q)*reject))


@lru_cache(maxsize=1)
def sensitivity_table():
    table = []
    # Search a grid of independent worlds, in increments of 10. No pilot outcomes used.
    for q in (.05,.10):
        for delta in (.01,.02,.05):
            lo, hi = 0, 5
            while exact_scenario_power(10*hi,q,delta) < .80:
                lo,hi = hi,2*hi
            while hi-lo > 1:
                mid = (hi+lo)//2
                if exact_scenario_power(10*mid,q,delta) >= .80:
                    hi = mid
                else:
                    lo = mid
            table.append({"method":"independent_world_single_target_exact_McNemar_scenario",
                          "discordance":q, "target_pp":delta*100, "worlds_for_80pct":10*hi,
                          "power_at_suggested_n":exact_scenario_power(10*hi,q,delta),
                          "power_at_24":exact_scenario_power(24,q,delta)})
    return table


def analyze(study, main, forks):
    frames = {(r["world"],r["target"],r["context"],r["arm"]):r for r in main}
    expected = 4*len(study.worlds)*len(study.arms)
    expected_forks = 2*min(8,len(study.worlds))*len(study.arms)
    conditions, effects, planning, swaps, zero_bounds = [], [], [], [], []
    power_metrics = {}

    # Only combine conditions/arms on the same world; every contrast requires BOTH target requests.
    def contrast(name, terms, unit="pp", plan=False):
        # Each term is (coefficient, arm, context, field).
        vals = []
        for w in study.worlds:
            pair = []
            for target in (0,1):
                indices = [(coef,frames.get((w["id"],target,c,a)),m) for coef,a,c,m in terms]
                if all(row is not None and m in row for _,row,m in indices):
                    pair.append(sum(coef*float(row[m]) for coef,row,m in indices))
            if len(pair)==2:
                vals.append(sum(pair)/2*(100 if unit=="pp" else 1))
        effects.append(effect_row(name,vals,unit))
        if plan:
            power_metrics[name] = (vals,unit)

    def shift(arm, metric):
        return [(1,arm,"C1",metric),(-1,arm,"C0",metric)]

    def minus(terms):
        return [(-coef,a,c,m) for coef,a,c,m in terms]

    summary_metrics = ("first_lookup_gold","first_lookup_other","first_valid_lookup",
         "attempted_conflict_reservation_item","accepted_conflict_reservation_item",
         "any_order_conflict_value","accepted_conflict_value","strict_task_success",
         "no_accepted_order","format_failure","tool_error","prompt_budget_hit","action_budget_hit","model_calls","output_tokens","generation_seconds",
         "gold_logp","other_logp","margin")
    for arm in study.arms:
        for ctx in ("C0","C1"):
            rows = [r for r in main if r["arm"]==arm and r["context"]==ctx]
            if rows:
                summary = {"arm":arm,"context":ctx,"episodes":len(rows)}
                fields = list(summary_metrics) + (["direct_valid","direct_correct","direct_other"] if arm=="none" else [])
                for m in fields:
                    summary[m+"_mean"] = float(np.mean([r[m] for r in rows]))
                    if isinstance(rows[0][m],bool):
                        summary[m+"_count"] = sum(r[m] for r in rows)
                conditions.append(summary)
        contrast("mechanism_drop_C0_minus_C1/"+arm, minus(shift(arm,"margin")), "log_odds",plan=arm=="none")
        contrast("wrong_reservation_C1_minus_C0/"+arm,shift(arm,"accepted_conflict_reservation_item"),
                 plan=arm=="none")
        contrast("first_wrong_lookup_C1_minus_C0/"+arm,shift(arm,"first_lookup_other"))
        contrast("wrong_accepted_order_C1_minus_C0/"+arm,shift(arm,"accepted_conflict_value"))
        pairs = [[frames[(w["id"],t,"C1",arm)] for t in (0,1)] for w in study.worlds
                 if all((w["id"],t,"C1",arm) in frames for t in (0,1))]
        if pairs:
            n = len(pairs)
            swaps.append({"arm":arm,"complete_worlds":n,"both_targets_strict":sum(all(r["strict_task_success"] for r in pair) for pair in pairs),
                          "both_first_lookup_gold":sum(all(r["first_lookup_gold"] for r in pair) for pair in pairs)})
            k = sum(any(r["accepted_conflict_reservation_item"] for r in pair) for pair in pairs)
            from scipy.stats import beta
            upper = 1.0 if k==n else float(beta.ppf(.95,k+1,n-k))
            zero_bounds.append({"arm":arm,"event":"one_or_both_targets_wrong_reservation_under_C1",
                                "worlds":n,"worlds_with_event":k,"upper_one_sided_95":upper})
    for arm in ("none", *[a for a in study.arms if a.startswith("random")]):
        contrast("mechanism_drop_reduction_"+arm+"_minus_global80",
                 minus(shift(arm,"margin"))+shift("global80","margin"),"log_odds",plan=arm=="none")
        contrast("C1_margin_global80_minus_"+arm,[(1,"global80","C1","margin"),(-1,arm,"C1","margin")],"log_odds")
        contrast("wrong_reservation_context_effect_"+arm+"_minus_global80",
                 shift(arm,"accepted_conflict_reservation_item")+minus(shift("global80","accepted_conflict_reservation_item")),
                 plan=arm=="none")
    contrast("C0_strict_success_global80_minus_none",[(1,"global80","C0","strict_task_success"),(-1,"none","C0","strict_task_success")])
    contrast("direct_first_choice_C1_minus_C0",shift("none","direct_other"))
    contrast("direct_minus_actor_context_effect_on_first_SKU",
             shift("none","direct_other")+minus(shift("none","first_lookup_other")),plan=True)
    # Positive gap means a larger change in wrong-SKU selections for the direct protocol.
    # Invalid outputs remain in denominators. Separately report direct_valid / first_valid_lookup.
    fork_pairs = {(r["world"],r["arm"],r["stocked"]):r for r in forks}
    fork_report = []
    for arm in study.arms:
        pairs = [[fork_pairs[(w["id"],arm,s)] for s in (True,False)] for w in study.worlds[:8]
                 if all((w["id"],arm,s) in fork_pairs for s in (True,False))]
        if pairs:
            fork_report.append({"arm":arm,"paired_worlds":len(pairs),
                 "stocked_correct":sum(p[0]["strict_task_success"] for p in pairs),
                 "out_of_stock_correct":sum(p[1]["strict_task_success"] for p in pairs),
                 "both_stock_branches_correct":sum(all(r["strict_task_success"] for r in p) for p in pairs)})
    for name,(vals,unit) in power_metrics.items():
        if study.worlds[0]["split"]!="pilot":
            continue
        n=len(vals)
        informative=sum(v!=0 for v in vals)
        # A small pilot can estimate variance very poorly. Never produce n=0 or promote a partial pilot.
        ready = n==len(study.worlds) and n>=24 and informative>=6 and np.std(vals,ddof=1)>0
        for target in ((1,2,5) if unit=="pp" else (.25,.5,1.0)):
            size = None
            if ready:
                z = NormalDist().inv_cdf(.975)+NormalDist().inv_cdf(.80)
                size = max(2,math.ceil((z*np.std(vals,ddof=1)/target)**2))
            planning.append({"method":"pilot_cluster_SD_normal_approximation","contrast":name,"worlds":n,
                             "nonzero_net_worlds":informative,"target_effect":target,"effect_unit":unit,
                             "target_pp":target if unit=="pp" else None,"worlds_for_80pct":size,
                             "status":"rough planning only" if ready else "insufficient pilot information"})
    planning.extend(sensitivity_table())
    complete = len(main)==expected and len(forks)==expected_forks
    return {"status":"COMPLETE" if complete else "PARTIAL","independent_worlds_planned":len(study.worlds),
            "main_units":[len(main),expected],"fixed_history_units":[len(forks),expected_forks],
            "primary_mechanism":"C0-C1 paired whole-SKU log odds drop; compare global80 with none and matched random",
            "primary_action":"tool-accepted reservation of other request SKU, one indicator per autonomous episode",
            "conditions":conditions,"effects":effects,"target_swap":swaps,"stock_fork":fork_report,
            "world_event_bounds":zero_bounds,"planning":planning,
            "notes":["Intervals resample whole worlds (3000 draws); constant samples have no bootstrap CI. Rare-event CIs are descriptive.",
                     "Formal sample-size scenarios concern one prespecified target per world; they are not power guarantees for the cluster interaction.",
                     "Stage planning uses full paired-world SD and an external target difference, not observed power. Small or sparse pilots leave N blank.",
                     "Incomplete contrasts use only fully paired worlds. All observed attempts, invalid outputs and denominators remain in main.csv.",
                     "Direct/agent comparison concerns the first SKU under two different prompts using the same weights. Stock forks have a forced first lookup."]}
'''


In [10]:
V61_ALIGNED_SOURCE = r'''"""v6.1 conditions and interventions in the frozen v6 namespace."""
from collections import Counter, defaultdict
from types import SimpleNamespace


CONDITIONS = ("C0", "Clex", "C1")
ITEM_PREFIX = '{"item":"sku-'
v6_records_for = records_for


def records_for(world, target, condition):
    if condition != "Clex":
        return v6_records_for(world, target, condition)
    records = v6_records_for(world, target, "C1")
    other_id = world["requests"][1-target]
    next(r for r in records if r["entity"] == other_id)["attribute"] = "preview_item"
    return records


def decision_tokens(backend, gold, other):
    """Tokenize output separately from the prompt, exactly as autoregressive decoding starts."""
    enc = lambda s: list(backend.tokenizer.encode(s, add_special_tokens=False))
    stem = enc(ITEM_PREFIX)
    texts = ['{"item":"' + sku for sku in (gold, other)]
    ids = [enc(text) for text in texts]
    if len(ids[0]) != len(ids[1]) or any(seq[:len(stem)] != stem for seq in ids):
        raise ValueError("Unmatched SKU tokenization or retokenized decision stem")
    for text, seq in zip(texts, ids):
        if backend.tokenizer.decode(seq, skip_special_tokens=False) != text:
            raise ValueError("Output token round trip failed")
    split = next((i for i,(a,b) in enumerate(zip(*ids)) if a != b), None)
    if split is None or split < len(stem):
        raise ValueError("Expected distinct numeric SKU values")
    return stem, ids, split


def prompts_for(world, target, condition):
    case = case_for(world, target)
    records = records_for(world, target, condition)
    return case, records, {"actor": actor_prompt(case, records, []),
                           "direct": direct_prompt(world, target, condition)}


def audit_aligned_inputs(backend, worlds, prior):
    max_tokens = 0
    for w in worlds:
        for t in (0, 1):
            gold, other = w["items"][t], w["items"][1-t]
            _, candidates, _ = decision_tokens(backend, gold, other)
            texts = {}
            for c in CONDITIONS:
                case, recs, texts[c] = prompts_for(w,t,c)
                if c != "Clex":
                    prev = prior[(w["id"],t,c)]
                    if (prev["actor"]["events"][0]["prompt"] != texts[c]["actor"] or
                            prev["direct"]["prompt"] != texts[c]["direct"] or
                            prev["records"] != recs or prev["actor"]["initial_state"] != case["initial_env"]):
                        raise ValueError("v6 prompt, records or environment changed: "+w["id"])
                for prompt in texts[c].values():
                    if (prompt.count(gold),prompt.count(other)) != (1,int(c!="C0")):
                        raise ValueError("Unexpected SKU exposure: "+w["id"])
                    max_tokens = max(max_tokens,len(backend.encode(prompt))+len(candidates[0]))
            for role in ("actor","direct"):
                neutral, exposure, relevant = [texts[c][role] for c in CONDITIONS]
                if exposure.count('"attribute":"preview_item"') != 1 or exposure.replace(
                        '"attribute":"preview_item"', '"attribute":"preferred_item"') != relevant:
                    raise ValueError("Clex/C1 must differ only in the other request's relation")
                seq = [backend.encode(x) for x in (neutral,exposure,relevant)]
                if len(set(map(len,seq))) != 1 or sum(a!=b for a,b in zip(seq[1],seq[2])) != 1:
                    raise ValueError("Clex/C1 must substitute exactly one token at the same position")
    if max_tokens > backend.cfg["max_prompt_tokens"]:
        raise ValueError("Initial input exceeds original context budget")
    return {"worlds":len(worlds),"max_tokens_including_candidate":max_tokens,
            "old_prompt_pairs_checked":len(prior), "equal_length_and_position":True}


def local_arms(ranking):
    top = [tuple(h) for h in ranking[:10]]
    if len(set(top)) != 10 or len(set(map(tuple,ranking))) != 80:
        raise ValueError("Expected the original ordered 80-head ranking")
    arms = {"none":[], "top10":top}
    for draw in range(3):
        rng = random.Random(61401 + draw)
        chosen = []
        for layer,n in sorted(Counter(l for l,h in top).items()):
            pool = [h for h in range(32) if (layer,h) not in set(map(tuple,ranking))]
            chosen.extend((layer,h) for h in rng.sample(pool,n))
        arms[f"random10_{draw+1}"] = chosen
    return arms


def tl_cache(backend):
    from transformer_lens.past_key_value_caching import HookedTransformerKeyValueCache
    return HookedTransformerKeyValueCache.init_cache(backend.model.cfg, backend.device, batch_size=1)


def aligned_score(backend, prompt, gold, other, heads=(), cache_factory=tl_cache):
    """Original one-sequence cached decoder. Score only SKU digits/hyphens, never JSON punctuation."""
    stem, choices, split = decision_tokens(backend,gold,other)
    prompt_tokens = backend.tokens(prompt)
    if prompt_tokens.shape[1] + len(choices[0]) > backend.cfg["max_prompt_tokens"]:
        raise ValueError("Probe exceeds input budget")
    scores = []
    key_prefix = choices[0][:split]
    observed_prefix = backend.tokenizer.decode(key_prefix, skip_special_tokens=False)
    info = {"prefix_text":observed_prefix, "decision_output_offset":split,
            "shared_output_ids":key_prefix, "sku_tokens":len(choices[0])-len(stem),
            "gold_decision_id":choices[0][split],"other_decision_id":choices[1][split]}
    with torch.inference_mode():
        for branch, out_ids in enumerate(choices):
            cache = cache_factory(backend)
            step_tokens = prompt_tokens
            total = 0.0
            mismatch = None
            for i, token in enumerate(out_ids):
                # i is the index of the token predicted by THIS forward pass.
                hooks = ablation_hooks(backend.model, heads) if heads and i==split else []
                if hooks and step_tokens.shape[1] != 1:
                    raise RuntimeError("A local mask may touch only one query position")
                with backend.model.hooks(fwd_hooks=hooks):
                    logits = backend.model(step_tokens,past_kv_cache=cache,return_type="logits")[0,-1].float()
                if branch == 0 and i <= split:
                    argmax = int(logits.argmax().item())
                    if i < split and argmax != token and mismatch is None:
                        mismatch = i
                    if i == split:
                        info.update(first_margin=float((logits[out_ids[i]]-logits[choices[1][i]]).item()),
                                    greedy_text=backend.tokenizer.decode(key_prefix+[argmax],skip_special_tokens=False),
                                    greedy_decision_id=argmax, replay_prefix_match=mismatch is None,
                                    first_prefix_mismatch=mismatch)
                if i >= len(stem):
                    total += float(logits.log_softmax(-1)[token].item())
                step_tokens = torch.tensor([[token]], device=backend.device, dtype=torch.long)
            scores.append(total)
            del cache
    return dict(info, gold_sku_logp=scores[0], other_sku_logp=scores[1], sku_margin=scores[0]-scores[1])


class FirstActionMask:
    """Delegate generation, stopping and tool execution to v6. Arm one cached forward only."""
    def __init__(self, backend, prompt, common_ids, heads):
        self.backend, self.prompt, self.common, self.heads = backend,prompt,common_ids,heads
        self.cfg, self.encode = backend.cfg,backend.encode
        self.fired = False
        self.used = False

    def generate(self, prompt, role, heads, use_cache=True):
        if self.used:
            return self.backend.generate(prompt,role=role,heads=heads,use_cache=use_cache)
        if prompt!=self.prompt or not self.common or heads or not use_cache or role!="actor":
            raise ValueError("Local intervention requires the original first free Actor call")
        self.used = True
        state = SimpleNamespace(prefilled=False, offset=0, matching=True, active=False)
        model = self.backend.model
        layer_hits = Counter()
        frozen_prompt = self.backend.tokens(prompt)

        def before(module, args):
            tokens = args[0]
            state.active = False
            if not state.prefilled:
                if not torch.equal(tokens,frozen_prompt):
                    raise RuntimeError("Unexpected prefill in local decode")
                state.prefilled = True
            elif state.matching and state.offset < len(self.common):
                if tuple(tokens.shape) != (1,1):
                    raise RuntimeError("Expected one cached decode token")
                state.matching = int(tokens.item()) == self.common[state.offset]
                state.offset += 1
                state.active = state.matching and state.offset == len(self.common)
                if state.active:
                    self.fired = True

        def gated(name, fn):
            def apply(value, hook):
                if not state.active:
                    return value
                if value.shape[1] != 1:
                    raise RuntimeError("Unscoped position mask")
                layer_hits[name] += 1
                return fn(value,hook)
            return apply

        hooks = [(name,gated(name,fn)) for name,fn in ablation_hooks(model,self.heads)]
        handle = model.register_forward_pre_hook(before)
        try:
            with model.hooks(fwd_hooks=hooks):
                out = self.backend.generate(prompt,role=role,heads=(),use_cache=True)
        finally:
            handle.remove()
        if not state.prefilled or any(layer_hits[name] != int(self.fired) for name,_ in hooks):
            raise RuntimeError("Free-action intervention did not have its declared coverage")
        return out
'''


In [11]:
V61_ALIGNED_STUDY_SOURCE = '"""Resume, summarize and export the developmental v6.1 pilot."""\nimport zipfile\nimport zlib\nimport base64\n\n\ndef read_reference(encoded, expected_sha):\n    raw = zlib.decompress(base64.b64decode(encoded, validate=True))\n    if hashlib.sha256(raw).hexdigest() != expected_sha:\n        raise ValueError("Built-in v6 source data failed its SHA-256 check")\n    data = json.loads(raw)\n    manifest = data["manifest"]\n    report = data["coverage"]\n    if (manifest["version"] != "v6_truthful_binding_1" or manifest["stage"] != "pilot" or\n            report != {"status":"COMPLETE","main_units":[288,288],"fixed_history_units":[48,48]} or\n            manifest["worlds"] != make_worlds("pilot",24)):\n        raise ValueError("This follow-up requires the complete, original 24-world pilot")\n    signature = stable_hash(manifest)\n    prior = {}\n    for w in manifest["worlds"]:\n        for t in (0,1):\n            for c in ("C0","C1"):\n                key = f"main/{w[\'id\']}/none-{t}-{c}"\n                obj = data["units"][key]\n                r = obj["result"]\n                if (obj["signature"] != stable_hash([signature,key]) or obj["digest"] != stable_hash(r) or\n                        (r["world"],r["target"],r["context"],r["arm"]) != (w["id"],t,c,"none")):\n                    raise ValueError("Prior unit identity or digest mismatch: " + key)\n                prior[(w["id"],t,c)] = r\n    if len(prior) != len(data["units"]):\n        raise ValueError("Unexpected prior results")\n    return manifest, prior\n\n\ndef actor_actions(actor):\n    return [(e["generation"]["text"],e["action"],e["observation"]) for e in actor["events"]]\n\n\nclass AlignedStudy(Study):\n    def __init__(self, root, backend, reference, prior, identity, verify, scorer=aligned_score):\n        super().__init__(root,backend,reference["worlds"],[],local_arms(reference["arms"]["global80"]),\n                         identity,verify,scorer)\n        self.reference, self.prior = reference,prior\n\n    def prepare_followup(self):\n        self.check()\n        audit = self.unit("checks/inputs",lambda: audit_aligned_inputs(self.backend,self.worlds,self.prior))\n\n        def bridge():\n            if self.manifest["runtime"] != self.reference["runtime"]:\n                return {"reuse":False,"reason":"runtime fingerprint differs","sentinels":[]}\n            rows = []\n            # Defined by index/condition, independently of earlier errors or probe values.\n            for index,t,c in ((0,0,"C0"),(1,1,"C1")):\n                w = self.worlds[index]\n                old = self.prior[w["id"],t,c]\n                case,records,_ = prompts_for(w,t,c)\n                actor = run_actor(self.backend,case,records,[])\n                direct = direct_choice(self.backend,w,t,c)\n                rows.append(dict(world=w["id"],target=t,context=c,\n                                 actor_match=actor_actions(actor)==actor_actions(old["actor"]),\n                                 direct_match=direct["generation"]["text"]==old["direct"]["generation"]["text"]))\n            ok = all(s["actor_match"] and s["direct_match"] for s in rows)\n            return {"reuse":ok,"reason":"fingerprint and sentinels match" if ok else "sentinel output differs",\n                    "sentinels":rows}\n        reuse = self.unit("checks/bridge",bridge)\n        print(" Pair & Length Check ",audit)\n        print(" Reuse Old Behaviour ",reuse,flush=True)\n        if not reuse["reuse"]:\n            print(" This time regenerates  96  Group  C0/C1  There is no intervention, and pairings are made using the current fixed model. ")\n        return reuse\n\n    def compute_behavior(self,w,t,c,arm,role,reuse):\n        case,records,prompts = prompts_for(w,t,c)\n        if arm=="none" and c!="Clex" and reuse:\n            orig = self.prior[w["id"],t,c]\n            return {"output":orig[role],"origin":"v6_archive","mask_fired":False,\n                    "v6_unit":f"main/{w[\'id\']}/none-{t}-{c}","v6_result_digest":stable_hash(orig)}\n        if role=="direct":\n            if arm!="none":\n                raise ValueError("The localized head intervention applies only to the Actor")\n            output = direct_choice(self.backend,w,t,c)\n            return {"output":output,"origin":"new","mask_fired":False}\n        be = self.backend\n        if arm!="none":\n            _, ids, split = decision_tokens(be,w["items"][t],w["items"][1-t])\n            be = FirstActionMask(be,prompts["actor"],ids[0][:split],self.arms[arm])\n        output = run_actor(be,case,records,[])\n        return {"output":output,"origin":"new","mask_fired":arm!="none" and be.fired}\n\n    def run_aligned(self, arms, limit=None):\n        if limit is not None and (type(limit) is not int or limit<=0):\n            raise ValueError("WORLDS_PER_CALL must be positive or None")\n        self.check()\n        bridge = self.unit("checks/bridge")\n        if bridge is None or self.unit("checks/inputs") is None:\n            raise RuntimeError("Run the input and archive checks first")\n        for arm in arms:\n            done = 0\n            for w in self.worlds:\n                before = self.new_units\n                for t in (0,1):\n                    for c in CONDITIONS:\n                        for role in (("actor","direct") if arm=="none" else ("actor",)):\n                            key = f"{arm}/{w[\'id\']}/{t}-{c}-{role}"\n                            self.unit("behavior/"+key,lambda:self.compute_behavior(w,t,c,arm,role,bridge["reuse"]))\n                            def score():\n                                p = prompts_for(w,t,c)[2][role]\n                                return self.scorer(self.backend,p,w["items"][t],w["items"][1-t],self.arms[arm])\n                            self.unit("probe/"+key,score)\n                if self.new_units>before:\n                    done += 1\n                    print(f"[{arm}] {w[\'id\']} complete | {self.new_units} new checkpoints",flush=True)\n                    if limit is not None and done>=limit:\n                        break\n        self.check()\n\n    def collect_aligned(self, arms):\n        rows = []\n        for arm in arms:\n            for w in self.worlds:\n                for t in (0,1):\n                    for c in CONDITIONS:\n                        for role in (("actor","direct") if arm=="none" else ("actor",)):\n                            key = f"{arm}/{w[\'id\']}/{t}-{c}-{role}"\n                            b,q = self.unit("behavior/"+key),self.unit("probe/"+key)\n                            if b is None or q is None:\n                                continue\n                            out = b["output"]\n                            raw = (out["events"][0]["generation"]["text"] if out["events"] else "") if role=="actor" else out["generation"]["text"]\n                            row = dict(world=w["id"],target=t,template=w["template"],\n                                       target_position=(1-t if w["reverse"] else t)+1,context=c,role=role,arm=arm,\n                                       origin=b["origin"],mask_fired=b["mask_fired"],**q)\n                            row["text_prefix_match"] = raw.startswith(q["prefix_text"])\n                            row["on_path"] = row["text_prefix_match"] and q["replay_prefix_match"]\n                            row["decision_agrees_with_trace"] = row["on_path"] and raw.startswith(q["greedy_text"])\n                            if role=="actor":\n                                row.update(out["metrics"])\n                                row.update(first_other=out["metrics"]["first_lookup_other"],\n                                           first_gold=out["metrics"]["first_lookup_gold"],\n                                           first_valid=out["metrics"]["first_valid_lookup"],terminal=out["terminal"])\n                                if c!="Clex" and arm=="none":\n                                    row["archived_tool_first_margin"] = self.prior[w["id"],t,c]["probe"]["margin"]\n                            else:\n                                row.update(first_other=out["other"],first_gold=out["correct"],\n                                           first_valid=out["valid"],schema_valid=out["schema_valid"])\n                            rows.append(row)\n        return rows\n\n\ndef summarize_aligned(study, arms, rows):\n    table = {(r["arm"],r["role"],r["world"],r["target"],r["context"]):r for r in rows}\n    expected = len(study.worlds)*2*len(CONDITIONS)*(len(arms)+1)\n    groups,effects,world_values = [],[],[]\n    common = ("on_path","decision_agrees_with_trace","first_valid","first_gold","first_other",\n              "first_margin","sku_margin","gold_sku_logp","other_sku_logp")\n    actions = ("attempted_conflict_reservation_item","accepted_conflict_reservation_item",\n               "any_order_conflict_value","accepted_conflict_value","strict_task_success",\n               "format_failure","tool_error","action_budget_hit","no_accepted_order","mask_fired")\n    for arm in arms:\n        for role in (("actor","direct") if arm=="none" else ("actor",)):\n            for c in CONDITIONS:\n                group = [r for r in rows if (r["arm"],r["role"],r["context"])==(arm,role,c)]\n                if not group:\n                    continue\n                summary = dict(arm=arm,role=role,context=c,n=len(group),\n                               from_v6=sum(r["origin"]=="v6_archive" for r in group))\n                for metric in common + (actions if role=="actor" else ()):\n                    values = [r[metric] for r in group]\n                    summary[metric+"_mean"] = float(np.mean(values))\n                    if isinstance(values[0],bool):\n                        summary[metric+"_count"] = sum(values)\n                groups.append(summary)\n\n    def add(label, metric, terms, require_path=False):\n        # All cells and both target requests must exist in the SAME world.\n        vals = []\n        is_margin = metric in ("first_margin","sku_margin","gold_sku_logp","other_sku_logp")\n        for w in study.worlds:\n            pair = []\n            for t in (0,1):\n                block = [(coef,table.get((a,p,w["id"],t,c))) for coef,a,p,c in terms]\n                if all(row is not None and (not require_path or row["on_path"]) for _,row in block):\n                    pair.append(sum(coef*float(row[metric]) for coef,row in block))\n            if len(pair)==2:\n                v = sum(pair)/2*(1 if is_margin else 100)\n                vals.append(v)\n                world_values.append({"world":w["id"],"contrast":label,"metric":metric,"value":v,\n                                     "subset":"on_path" if require_path else "all"})\n        effects.append(dict(effect_row(label,vals,"natural_log" if is_margin else "pp"),metric=metric,\n                            subset="on_path" if require_path else "all"))\n\n    for arm in arms:\n        for role in (("actor","direct") if arm=="none" else ("actor",)):\n            for left,right in (("C0","C1"),("C0","Clex"),("Clex","C1")):\n                label = f"{arm}/{role}/{left}-{right}"\n                terms = [(1,arm,role,left),(-1,arm,role,right)]\n                for metric in ("first_margin","sku_margin","gold_sku_logp","other_sku_logp"):\n                    add(label,metric,terms)\n                    if metric in ("first_margin","sku_margin"):\n                        add(label,metric,terms,True)\n                for metric in ("first_other",) + (("accepted_conflict_reservation_item","accepted_conflict_value")\n                                                  if role=="actor" else ()):\n                    add(f"{arm}/{role}/{right}-{left}",metric,[(-coef,a,p,c) for coef,a,p,c in terms])\n    for metric in ("first_margin","sku_margin","first_other"):\n        # Larger margin drop or wrong-choice increase in the direct protocol.\n        sign = -1 if metric=="first_other" else 1\n        add("direct_minus_actor/additional_relation",metric,\n            [(sign,"none","direct","Clex"),(-sign,"none","direct","C1"),\n             (-sign,"none","actor","Clex"),(sign,"none","actor","C1")])\n    utility = []\n    if "top10" in arms:\n        for a in [x for x in arms if x!="top10"]:\n            for metric in ("first_margin","sku_margin"):\n                add(f"extra_drop/{a}-top10",metric,[(1,a,"actor","Clex"),(-1,a,"actor","C1"),\n                    (-1,"top10","actor","Clex"),(1,"top10","actor","C1")])\n            add(f"clean_success/top10-{a}","strict_task_success",\n                [(1,"top10","actor","C0"),(-1,a,"actor","C0")])\n        for g in groups:\n            if g["role"]=="actor" and g["context"]=="C0":\n                utility.append({"arm":g["arm"],"n":g["n"],"strict_success":g["strict_task_success_count"],\n                                "clean_utility_pass":g["n"]==2*len(study.worlds) and g["strict_task_success_mean"]>=.9})\n    report = {"status":"COMPLETE" if len(rows)==expected else "PARTIAL","completed_expected":[len(rows),expected],\n              "scope":"local" if len(arms)>1 else "baseline","independent_worlds":len(study.worlds),\n              "arms":arms,"conditions":groups,"effects":effects,"clean_utility":utility,\n              "archive_reuse":study.unit("checks/bridge"),"input_audit":study.unit("checks/inputs"),\n              "notes":["All paired contrasts average A/B within world; 3000 bootstrap draws resample worlds.",\n                       "Clex/C1 hold both SKUs and positions fixed; C0/Clex also changes the attribute and lot/SKU.",\n                       "All includes standardized probes for off-path formats. on_path requires matching recorded text and cached greedy prefix.",\n                       "Direct and Actor use different prompts. Autonomous actions always include failures.",\n                       "These are the 24 developmental worlds from v6. A constant sample has no bootstrap interval."]}\n    return report,world_values\n\n\ndef local_ready(report):\n    if report["scope"]!="baseline" or report["status"]!="COMPLETE":\n        return False\n    contrast = next(e for e in report["effects"] if (e["contrast"],e["metric"],e["subset"])==\n                    ("none/actor/Clex-C1","first_margin","all"))\n    relevant = [g for g in report["conditions"] if g["role"]=="actor" and g["context"] in ("Clex","C1")]\n    return contrast["estimate"]>0 and all(g["on_path_mean"]>=.9 for g in relevant)\n\n\ndef export_aligned(study, arms):\n    study.check()\n    rows = study.collect_aligned(arms)\n    report,world_values = summarize_aligned(study,arms,rows)\n    dest = study.root/"reports"/report["scope"]\n    for name,values in (("scores-actions",rows),("conditions",report["conditions"]),\n                        ("paired-effects",report["effects"]),("paired-worlds",world_values)):\n        write_csv(dest/(name+".csv"),values)\n    save_json(dest/"report.json",report)\n    archive = study.root.parent/("v6p1_"+report["scope"]+"_"+report["status"]+"_results.zip")\n    pending = archive.with_suffix(".zip.pending")\n    with zipfile.ZipFile(pending,"w",compression=zipfile.ZIP_DEFLATED) as z:\n        z.write(study.root/"manifest.json","manifest.json")\n        for directory in [dest,study.root/"checks",*[study.root/s/a for s in ("behavior","probe") for a in arms]]:\n            for file in sorted(directory.rglob("*")):\n                if file.is_file() and file.suffix in (".json",".csv"):\n                    z.write(file,file.relative_to(study.root))\n    with zipfile.ZipFile(pending) as z:\n        if z.testzip() is not None or json.loads(z.read(f"reports/{report[\'scope\']}/report.json"))!=report:\n            raise IOError("Result archive did not verify; runtime must remain connected")\n    os.replace(pending,archive)\n    return report,archive\n'


In [12]:
#@title  Freezing  v6  Actual results (compressed)  JSON  data) 
V61_ZIP_SHA = '50e82fcc84eaa2df6b0eebc78e5d9a1ceda86a1e322f07780c0b05b3c1b99894'
V61_ZIP_URL = 'https://drive.google.com/file/d/1mboKQSO0yujvyGH7zq0vu_-pPT3XQ6gH/view'
V61_DATA_SHA = '19c3a3579078af4ecc29d74aa70c6c2202a8dbdd93a9adf5f6e66d76d5559fa8'
V61_BASE64 = (
'...'
)


In [13]:
V61_NS = dict(V6_RUNTIME_NAMESPACE)
V61_SOURCES = [V6_LEGACY_SOURCE,V6_TASKS_SOURCE,V6_STUDY_SOURCE,V6_REPORTS_SOURCE,V61_ALIGNED_SOURCE,V61_ALIGNED_STUDY_SOURCE]
for i,source in enumerate(V61_SOURCES):
    exec(compile(source,f"<v6p1-{i}>","exec"),V61_NS)
V61_HASHES = [hashlib.sha256(s.encode()).hexdigest() for s in V61_SOURCES]
V61_REFERENCE, V61_PRIOR = V61_NS["read_reference"](V61_BASE64,V61_DATA_SHA)
if V61_HASHES[:4]!=V61_REFERENCE["source_sha256"]:
    raise RuntimeError("The original v6 task or scoring definitions differ")
print(" Original check point verified ",len(V61_PRIOR)," individual (each inclusive)  Agent  and  Direct）")


 Original check point verified  96  individual (each inclusive)  Agent  and  Direct）


## 3 ·  Load Model and Reuse Check 

 New  Colab  sessions must reload the fixed model weights. Within an existing compatible kernel, this cell attempts to reuse the loaded model. Original  ZIP  Embedded, no search required  Drive  files or other uploads. If the software fingerprint or fixed replay differs, explicitly report it and regenerate in the current environment using the same model  C0/C1  No interference, no mix of two sets of output. 


In [15]:
v61_hash = V61_NS["stable_hash"]
stack = v61_hash([V6_PACKAGE_VERSIONS,sys.version.split()[0],V6_CFG,V6_CORE_SOURCE,V6_RUNTIME_SOURCE])[:16]
candidates = []
for name in ("V61_PARENT","V6_PARENT","V5_PARENT"):
    p = globals().get(name)
    if isinstance(p,dict) and "backend" in p and "MODEL_IDENTITY" in p:
        candidates.append({"label":name,"backend":p["backend"],"model_identity":p["MODEL_IDENTITY"]})
V61_PARENT = V6_RUNTIME_NAMESPACE["v4_load_runtime"](V6_CFG,BASE_DIR/"runtime"/stack,V6_PACKAGE_VERSIONS,
    V6_CORE_SOURCE,V6_RUNTIME_SOURCE,candidates=candidates)
V61_ID = dict(version="v6p1_aligned_exposure_1",runtime=V61_PARENT["RUN_FINGERPRINT"],
    model=V61_PARENT["MODEL_IDENTITY"],gpu=torch.cuda.get_device_name(V61_PARENT["backend"].device),
    source_sha256=V61_HASHES,prior_zip_sha256=V61_ZIP_SHA,prior_zip_url=V61_ZIP_URL,
    prior_payload_sha256=V61_DATA_SHA,unit="24 prior independent worlds, average repeated A/B within world",
    primary="untreated Actor first divergent token margin: Clex minus C1",
    secondaries="whole SKU log odds, Direct, world-paired protocol difference, all original action metrics",
    Clex="other request preview_item equals its preferred_item; one exposure in the same slot",
    local="fixed old top10, 3 layer-matched random sets; one query position of first free Actor action",
    local_gate="complete pilot, positive Actor Clex-C1 first-token drop; both contexts >=90% on path",
    utility="all arms retained; >=90% clean strict success")
V61_RUN = BASE_DIR/v61_hash(V61_ID)[:16]
if V61_RUN.name != "eb30b84321534641":
    print(" This environment and the completed  baseline  Unlike this, matching will be re-established in the independent catalogue. ",flush=True)
else:
    print(" Matched  v6.1 baseline  , read the existing checkpoints. ",flush=True)
V61_STUDY = V61_NS["AlignedStudy"](V61_RUN/"experiment",V61_PARENT["backend"],V61_REFERENCE,V61_PRIOR,
                                   V61_ID,V61_PARENT["verify_live_run"])
print(" model ",V61_PARENT["loader_reuse"]["mode"]," This checkpoint ",V61_RUN)
V61_STUDY.prepare_followup()


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Loaded pretrained model meta-llama/Llama-3.1-8B into HookedTransformer
 Matched  v6.1 baseline  , read the existing checkpoints. 
 model  load_pinned_model  This checkpoint  /content/drive/MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v6p1_aligned_sku/pilot-main/eb30b84321534641
 Pair & Length Check  {'worlds': 24, 'max_tokens_including_candidate': 2293, 'old_prompt_pairs_checked': 96, 'equal_length_and_position': True}
 Reuse Old Behaviour  {'reuse': True, 'reason': 'fingerprint and sentinels match', 'sentinels': [{'world': 'pilot-00001', 'target': 0, 'context': 'C0', 'actor_match': True, 'direct_match': True}, {'world': 'pilot-00002', 'target': 1, 'context': 'C1', 'actor_match': True, 'direct_match': True}]}


{'reuse': True,
 'reason': 'fingerprint and sentinels match',
 'sentinels': [{'world': 'pilot-00001',
   'target': 0,
   'context': 'C0',
   'actor_match': True,
   'direct_match': True},
  {'world': 'pilot-00002',
   'target': 1,
   'context': 'C1',
   'actor_match': True,
   'direct_match': True}]}

## 4 ·  Main experiment 

Actor  and  Direct  Use each of the original hints to prefix the same natural output  `{"item":"sku-`  The main indicator is the first different.  SKU token   logit  difference (correct minus competing). Next, for complete  SKU  compute the log-probability difference over remaining characters, without treating  JSON  Mark, tool name or follow-up is included. The probe follows the original  KV cache  PV  token  replay, checking whether each free greedy choice follows the shared prefix and verifying saved outputs. All trajectories remain in the behavioral denominator. Separately report position coverage and paired scores when both conditions reach it. 

 The main contrast is...  Actor   **Clex  Scores  − C1  Scores **， Positives indicate that the same exposure to preferred relationships further weakens the correctness.  SKU  The advantage. C0 − C1  Its a change. C0 − Clex  Includes changes in the visibility and properties of the commodity number. Direct  and  Actor  still uses different prompts. Report their difference as a within-scenario paired interaction. 


In [16]:
V61_STUDY.run_aligned(["none"],limit=WORLDS_PER_CALL)
BASE_REPORT, BASE_ZIP = V61_NS["export_aligned"](V61_STUDY,["none"])
REPORT, RESULTS_ZIP = BASE_REPORT, BASE_ZIP
print("baseline",REPORT["status"],REPORT["completed_expected"],"Drive ZIP",RESULTS_ZIP)
display(pd.DataFrame(REPORT["conditions"]))
display(pd.DataFrame([e for e in REPORT["effects"] if e["metric"]=="first_margin" and
                     (e["contrast"].endswith("Clex-C1") or e["contrast"].startswith("direct_minus"))]))


baseline COMPLETE [288, 288] Drive ZIP /content/drive/MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v6p1_aligned_sku/pilot-main/eb30b84321534641/v6p1_baseline_COMPLETE_results.zip


,arm,role,context,n,from_v6,on_path_mean,on_path_count,decision_agrees_with_trace_mean,decision_agrees_with_trace_count,first_valid_mean,...,format_failure_mean,format_failure_count,tool_error_mean,tool_error_count,action_budget_hit_mean,action_budget_hit_count,no_accepted_order_mean,no_accepted_order_count,mask_fired_mean,mask_fired_count
0,none,actor,C0,48,48,1.0,48,1.0,48,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,none,actor,Clex,48,0,1.0,48,1.0,48,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,none,actor,C1,48,48,1.0,48,1.0,48,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,none,direct,C0,48,48,1.0,48,1.0,48,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,none,direct,Clex,48,0,1.0,48,1.0,48,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,none,direct,C1,48,48,1.0,48,1.0,48,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,contrast,unit,n_worlds,estimate,ci95_low,ci95_high,positive_worlds,negative_worlds,zero_worlds,metric,subset
0,none/actor/Clex-C1,natural_log,24,2.414062,2.263021,2.562500,24,0,0,first_margin,all
1,none/actor/Clex-C1,natural_log,24,2.414062,2.263021,2.562500,24,0,0,first_margin,on_path
2,none/direct/Clex-C1,natural_log,24,1.265625,1.166667,1.367253,24,0,0,first_margin,all
3,none/direct/Clex-C1,natural_log,24,1.265625,1.166667,1.367253,24,0,0,first_margin,on_path
4,direct_minus_actor/additional_relation,natural_log,24,-1.148438,-1.312500,-0.986979,0,24,0,first_margin,all


## 5 ·  Run local interventions 

 This version is run by default. The starting condition is  baseline  Full, complete. Actor   Clex − C1  Head fork.  margin  At least a natural arrival rate for both conditions.  90%。 This is a pre-written development budget rule. Keep it open or not.  baseline  All results. 

 Before the original paper  10  Number of random head matching by layer, head and group, only in models ** Natural formation ** Common  SKU  The prefix and the upcoming selection of the first different  token  , block the corresponding position at once  query-head  outputs. The original generator still freely generates the complete action. Record no activation if it misses this position. Influence at this position can propagate through the current action  KV  cache continues propagating. The next tool call uses a fresh intact-model cache. Separately count  4 × 24 × 2 × 3 = 576  Article  Agent  Tracks and  576  A pair of probes. Check.  C0  strict task success, incorrect reservations and order attempts, and random-head controls. If neutral success is below  90%  as usual. 


In [ ]:
REQUESTED_FINISHED = False
REPORT, RESULTS_ZIP = BASE_REPORT, BASE_ZIP
if RUN_LOCAL and V61_NS["local_ready"](BASE_REPORT):
    arms = list(V61_STUDY.arms)
    V61_STUDY.run_aligned(arms[1:],limit=WORLDS_PER_CALL)
    REPORT, RESULTS_ZIP = V61_NS["export_aligned"](V61_STUDY,arms)
    display(pd.DataFrame(REPORT["clean_utility"]))
elif RUN_LOCAL:
    print("baseline  Not complete or not reached the predefined threshold.  baseline ZIP， Do not start a partial phase, do not break automatically. ")
REQUESTED_FINISHED = REPORT["status"]=="COMPLETE" and REPORT["scope"]==("local" if RUN_LOCAL else "baseline")
print(" Final Export ", REPORT["scope"], REPORT["status"], REPORT["completed_expected"], RESULTS_ZIP)

if REPORT["scope"]=="local" and REPORT["status"]=="COMPLETE":
    print(" Control Less  top-10  the value of the value of the  top-10  The group drops smaller. ")
    display(pd.DataFrame([e for e in REPORT["effects"] if e["contrast"].startswith("extra_drop/")
                          and e["metric"]=="first_margin" and e["subset"]=="all"]))


[top10] pilot-00001 complete | 12 new checkpoints
[top10] pilot-00002 complete | 24 new checkpoints
[top10] pilot-00003 complete | 36 new checkpoints
[top10] pilot-00004 complete | 48 new checkpoints
[top10] pilot-00005 complete | 60 new checkpoints
[top10] pilot-00006 complete | 72 new checkpoints
[top10] pilot-00007 complete | 84 new checkpoints
[top10] pilot-00008 complete | 96 new checkpoints
[top10] pilot-00009 complete | 108 new checkpoints
[top10] pilot-00010 complete | 120 new checkpoints
[top10] pilot-00011 complete | 132 new checkpoints
[top10] pilot-00012 complete | 144 new checkpoints
[top10] pilot-00013 complete | 156 new checkpoints
[top10] pilot-00014 complete | 168 new checkpoints
[top10] pilot-00015 complete | 180 new checkpoints
[top10] pilot-00016 complete | 192 new checkpoints
[top10] pilot-00017 complete | 204 new checkpoints
[top10] pilot-00018 complete | 216 new checkpoints
[top10] pilot-00019 complete | 228 new checkpoints
[top10] pilot-00020 complete | 240 new 

## 6 ·  validate  ZIP、 Synchronize to  Drive、 Release  GPU

 The baseline phase should be shown in its entirety  `baseline COMPLETE [288, 288]`。 This local phase should be shown in its entirety.  `local COMPLETE [864, 864]`。 Each unit is one. 「 Agreement  ×  Condition  ×  Objective  × head  Settings 」 behavior and paired probes. Reports, scenario-paired values, both raw-output types, masks, and provenance are included in the results  ZIP  Inside. PARTIAL  Can continue running. 


In [ ]:
import json, os, zipfile
from google.colab import drive, runtime
if AUTO_DISCONNECT and REQUESTED_FINISHED:
    target = Path(RESULTS_ZIP).resolve(strict=True)
    if not os.path.ismount("/content/drive") or not target.is_relative_to(Path("/content/drive/MyDrive").resolve()) or target.parent!=V61_RUN.resolve():
        raise RuntimeError(" The results have not yet been verified in this case  Drive  Output Directory, Keep  runtime")
    with zipfile.ZipFile(target) as z:
        saved = json.loads(z.read(f"reports/{REPORT['scope']}/report.json"))
        if (z.testzip() is not None or saved!=REPORT or saved["status"]!="COMPLETE" or
            json.loads(z.read("manifest.json"))!=json.loads(json.dumps(V61_STUDY.manifest))):
            raise RuntimeError("ZIP  Validation failed; retain  runtime")
    print("ZIP  Verifyed, synchronizing  Drive， Automatically release after success  GPU。",flush=True)
    drive.flush_and_unmount(timeout_ms=600_000)
    runtime.unassign()
else:
    print(" Result Location ",RESULTS_ZIP," Auto Disconnect ",AUTO_DISCONNECT," The requested phase has been completed ",REQUESTED_FINISHED)
